## 应用集成

### 1. 准备工作

#### 1.1 环境与数据集配置

- 环境安装

In [1]:
!git clone https://github.com/ultralytics/ultralytics.git
%cd ultralytics
# !pip install -r requirements.txt
!pip install -e .

Cloning into 'ultralytics'...
remote: Enumerating objects: 84090, done.
remote: Counting objects: 100% (557/557), done.
remote: Compressing objects: 100% (340/340), done.
remote: Total 84090 (delta 438), reused 226 (delta 217), pack-reused 83533 (from 3)
Receiving objects: 100% (84090/84090), 44.89 MiB | 9.50 MiB/s, done.
Resolving deltas: 100% (63448/63448), done.
/content/ultralytics
Obtaining file:///content/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.4.11-0.editable-py3-none-any.whl size=22973 sha256=8aa15e2200e015ed6a779e9e6bda00b39f08803f68fff3fd2c88cced4f772dd6
  Stored in directory: /tmp/pip-ephem-wheel-cache-f3yfhp1y/wheels/60/e0/59/e2f034f296abbdca5c21e3f5be76b9ca685f13c7bd17f8b58c
Succe

In [2]:
!pip install ninja packaging
!pip install flash-attn --no-build-isolation

# 验证安装
import flash_attn
print(f"FlashAttention version: {flash_attn.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 64.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=253780426 sha256=4e2f9e39313266b1544b68138b15b91ee6221eccf14f7902b7c6620351340810
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn
FlashAttention version: 2.8.3


In [3]:
from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 39.9/112.6 GB disk)


In [4]:
from ultralytics import YOLO

from IPython.display import display, Image
import torch
import torch.nn as nn

- 挂载谷歌云盘

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

- 下载数据集

In [4]:
# https://drive.google.com/file/d/1y6VgQxlUhBLC2qN_y-6elvz8U6BmftlI/view?usp=sharing  原始数据
# https://drive.google.com/file/d/1OiJ0YGK7Mj-ZCSNx_-smmDuD8qdrK9YR/view?usp=sharing  添加yaml文件数据
import gdown
file_id = '1OiJ0YGK7Mj-ZCSNx_-smmDuD8qdrK9YR'
# 下载文件
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'DeepPCB_YOLO.zip', quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1OiJ0YGK7Mj-ZCSNx_-smmDuD8qdrK9YR
From (redirected): https://drive.google.com/uc?id=1OiJ0YGK7Mj-ZCSNx_-smmDuD8qdrK9YR&confirm=t&uuid=7a59792b-1f41-4967-a0dd-1b0f61b1ae27
To: /content/ultralytics/DeepPCB_YOLO.zip
100%|██████████| 35.9M/35.9M [00:00<00:00, 45.1MB/s]


'DeepPCB_YOLO.zip'

In [5]:
# path: DeepPCB_YOLO 路径需要修改下
!unzip DeepPCB_YOLO.zip

Archive:  DeepPCB_YOLO.zip
   creating: DeepPCB_YOLO/
  inflating: DeepPCB_YOLO/test.txt   
  inflating: DeepPCB_YOLO/conversion_metadata.json  
  inflating: DeepPCB_YOLO/classes.txt  
   creating: DeepPCB_YOLO/images/
   creating: DeepPCB_YOLO/images/train/
  inflating: DeepPCB_YOLO/images/train/group13000_13000_13000050_test.jpg  
  inflating: DeepPCB_YOLO/images/train/group20085_20085_20085061_test.jpg  
  inflating: DeepPCB_YOLO/images/train/group20085_20085_20085096_test.jpg  
  inflating: DeepPCB_YOLO/images/train/group00041_00041_00041159_test.jpg  
  inflating: DeepPCB_YOLO/images/train/group00041_00041_00041046_test.jpg  
  inflating: DeepPCB_YOLO/images/train/group13000_13000_13000079_test.jpg  
  inflating: DeepPCB_YOLO/images/train/group77000_77000_77000010_test.jpg  
  inflating: DeepPCB_YOLO/images/train/group13000_13000_13000037_test.jpg  
  inflating: DeepPCB_YOLO/images/train/group00041_00041_00041139_test.jpg  
  inflating: DeepPCB_YOLO/images/train/group20085_20085_2

#### 1.2 模型搭建

- C3FFT模块

In [6]:
%%writefile /content/ultralytics/ultralytics/nn/modules/C3FFTFixed.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics.nn.modules.block import C3
from ultralytics.nn.modules.conv import Conv

class FourierConvLayerConservative(nn.Module):
    """保守的FFT层，变化更温和"""
    def __init__(self, channels):
        super().__init__()
        self.channels = channels

        # 更简单的频域处理
        self.freq_conv = nn.Conv2d(channels * 2, channels * 2, kernel_size=1)

        # 使用GroupNorm或InstanceNorm替代LayerNorm，它们对尺寸不敏感
        self.norm = nn.GroupNorm(8, channels * 2)  # 8组，适应各种尺寸

        # 可学习的缩放因子，初始值小
        self.scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, x):
        B, C, H, W = x.shape

        # FFT变换
        fft = torch.fft.rfft2(x, norm='ortho')
        real = fft.real
        imag = fft.imag

        # 拼接并处理
        combined = torch.cat([real, imag], dim=1)

        # 简单的频域变换
        combined = self.freq_conv(combined)

        # GroupNorm + 激活 (对尺寸不敏感)
        combined = self.norm(combined)
        combined = F.gelu(combined)  # GELU比ReLU更平滑

        # 分离并逆变换
        C_half = C
        real_out = combined[:, :C_half, :, :]
        imag_out = combined[:, C_half:, :, :]

        complex_out = torch.complex(real_out, imag_out)
        fft_result = torch.fft.irfft2(complex_out, s=(H, W), norm='ortho')

        # 应用缩放因子，控制FFT影响
        return self.scale * fft_result

class C3FFTFixed(nn.Module):
    """
    修复版C3FFT，采用残差连接和可控的FFT影响
    """
    def __init__(
        self,
        c1: int,
        c2: int,
        n: int = 1,
        shortcut: bool = True,
        g: int = 1,
        e: float = 0.5,
        fft_scale: float = 0.1
    ):
        super().__init__()

        # 主分支：标准C3
        self.c3_branch = C3(c1, c2, n=n, shortcut=shortcut, g=g, e=e)

        # 计算FFT分支的通道数
        fft_channels = max(c1 // 4, 16)

        # 轻量化的FFT分支
        self.fft_path = nn.Sequential(
            Conv(c1, fft_channels, k=1),  # 降维
            FourierConvLayerConservative(fft_channels),
            Conv(fft_channels, c2, k=1)   # 升维
        )

        # 可学习的FFT权重，初始值很小
        self.fft_weight = nn.Parameter(torch.tensor(fft_scale))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        残差融合：output = C3(x) + fft_weight * FFT(x)
        """
        # C3主分支
        c3_out = self.c3_branch(x)

        # FFT辅助分支
        fft_out = self.fft_path(x)

        # 可控的融合：C3主导，FFT辅助
        output = c3_out + self.fft_weight * fft_out

        return output

Writing /content/ultralytics/ultralytics/nn/modules/C3FFTFixed.py


- C3A2模块

In [7]:
%%writefile /content/ultralytics/ultralytics/nn/modules/C3A2.py

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from flash_attn import flash_attn_func
    HAS_FLASH_ATTN = True
except ImportError:
    HAS_FLASH_ATTN = False

from ultralytics.nn.modules.conv import Conv

class FlashAAttn(nn.Module):
    """
    Area-attention module with FlashAttention optimization.
    """
    def __init__(self, dim, num_heads=8, area=1, dropout=0.0):
        super().__init__()
        self.area = area
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        # 确保维度可整除
        if dim % num_heads != 0:
            raise ValueError(f"dim {dim} must be divisible by num_heads {num_heads}")

        # QKV投影
        self.qkv = Conv(dim, dim * 3, 1, act=False)
        self.proj = Conv(dim, dim, 1, act=False)

        # 位置编码
        self.pe = Conv(dim, dim, 3, 1, 1, g=dim, act=False)

        self.dropout = dropout

        # 动态判断是否使用FlashAttention
        self.use_flash = False  # 在forward中动态判断

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W

        # 动态判断是否使用FlashAttention
        use_flash = HAS_FLASH_ATTN and x.is_cuda and torch.cuda.is_available()

        # 生成QKV
        qkv = self.qkv(x)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # [B, num_heads, N, head_dim]

        # 应用区域分割
        if self.area > 1:
            q = q.reshape(B * self.area, self.num_heads, N // self.area, self.head_dim)
            k = k.reshape(B * self.area, self.num_heads, N // self.area, self.head_dim)
            v = v.reshape(B * self.area, self.num_heads, N // self.area, self.head_dim)
            B_eff = B * self.area
        else:
            B_eff = B

        # 使用FlashAttention或标准注意力
        if use_flash:
            try:
                # FlashAttention期望形状: [batch_size, seqlen, num_heads, head_dim]
                q = q.transpose(1, 2).contiguous()
                k = k.transpose(1, 2).contiguous()
                v = v.transpose(1, 2).contiguous()

                # 使用FlashAttention
                x = flash_attn_func(q, k, v, dropout_p=self.dropout, causal=False)

                # 恢复原始形状
                x = x.transpose(1, 2).reshape(B_eff, -1, H, W)
            except Exception as e:
                # 如果FlashAttention失败，回退到标准注意力
                #print(f"FlashAttention failed, falling back to standard attention: {e}")
                use_flash = False

        if not use_flash:
            # 标准注意力（备选）
            scale = self.head_dim ** -0.5
            attn = (q @ k.transpose(-2, -1)) * scale
            attn = F.softmax(attn, dim=-1)
            x = (attn @ v).transpose(1, 2).reshape(B_eff, -1, H, W)

        # 如果使用了区域分割，恢复原始批次大小
        if self.area > 1:
            x = x.reshape(B, -1, H, W)

        # 位置编码和投影
        pp = self.pe(x)
        return self.proj(x + pp)


class FlashABlock(nn.Module):
    """
    FlashAttention-based ABlock with memory optimization.
    """
    def __init__(self, dim, num_heads=8, area=1, mlp_ratio=1.0, dropout=0.0):
        super().__init__()

        # LayerNorm for pre-norm architecture
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

        # FlashAttention模块
        self.attn = FlashAAttn(dim, num_heads=num_heads, area=area, dropout=dropout)

        # MLP
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, dim),
            nn.Dropout(dropout)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # 保存输入用于残差连接
        identity = x

        # 预归一化 + 注意力
        B, C, H, W = x.shape
        x_norm = self.norm1(x.permute(0, 2, 3, 1).reshape(B, H*W, C))
        attn_out = self.attn(x_norm.reshape(B, H, W, C).permute(0, 3, 1, 2))
        x = identity + self.dropout(attn_out)

        # 预归一化 + MLP
        identity = x
        B, C, H, W = x.shape
        x_norm = self.norm2(x.permute(0, 2, 3, 1).reshape(B, H*W, C))
        mlp_out = self.mlp(x_norm).reshape(B, H, W, C).permute(0, 3, 1, 2)
        x = identity + self.dropout(mlp_out)

        return x


class C3A2(nn.Module):
    """
    C3 with FlashAttention blocks for maximum memory efficiency.
    """
    def __init__(self, c1, c2, n=1, shortcut=True, e=0.5, num_heads=8, area=1, dropout=0.0):
        super().__init__()

        # 隐藏通道数
        c_ = int(c2 * e)

        # 确保维度可整除
        if c_ % num_heads != 0:
            c_ = ((c_ + num_heads - 1) // num_heads) * num_heads

        # 卷积层
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c1, c_, 1, 1)
        self.cv3 = Conv(2 * c_, c2, 1)

        # FlashAttention blocks
        self.m = nn.Sequential(
            *(FlashABlock(c_, num_heads=num_heads, area=area, dropout=dropout)
              for _ in range(n))
        )

    def forward(self, x):
        y1 = self.m(self.cv1(x))
        y2 = self.cv2(x)
        return self.cv3(torch.cat((y1, y2), 1))

Writing /content/ultralytics/ultralytics/nn/modules/C3A2.py


- CGAFusion模块

In [8]:
%%writefile /content/ultralytics/ultralytics/nn/modules/CGAFusionFixed.py
import torch
import torch.nn as nn
import torch.nn.functional as F

__all__ = ["CGAFusion"]


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=3):
        super().__init__()
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x_cat))


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        reduced_channels = max(channels // reduction, 4)

        self.conv = nn.Sequential(
            nn.Conv2d(channels, reduced_channels, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(reduced_channels, channels, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y)
        return y


class PixelAttention(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        padding = (kernel_size - 1) // 2

        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size, padding=padding, groups=channels, bias=False),
            nn.Conv2d(channels, channels, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.conv(x)


class CGAFusionFixed(nn.Module):
    def __init__(self, c1, c2, reduction=8, alpha=0.65):
        super().__init__()

        # 保存原始参数用于调试
        self.original_c1 = c1
        self.original_c2 = c2
        self.reduction = reduction
        self.alpha = alpha

        # 关键修复：不再在__init__中创建卷积层，而是在第一次forward时创建
        # 这是因为YOLO在初始化时会用不同尺寸的输入多次调用forward
        self.conv1 = None
        self.conv2 = None
        self.spatial_attn = None
        self.channel_attn = None
        self.pixel_attn = None
        self.out_conv = None

        # 记录是否已初始化
        self._initialized = False

        #print(f"CGAFusion创建: c1={c1}, c2={c2}")

    def _initialize_once(self, feat1, feat2):
        """一次性初始化所有层（只在第一次forward时调用）"""
        if self._initialized:
            return

        # 确定统一的通道数
        self.unified_channels = min(feat1.shape[1], feat2.shape[1])

        # print(f"CGAFusion实际初始化: "
        #       f"feat1_channels={feat1.shape[1]}, feat2_channels={feat2.shape[1]}, "
        #       f"unified_channels={self.unified_channels}")

        # 创建通道调整卷积
        if feat1.shape[1] != self.unified_channels:
            self.conv1 = nn.Conv2d(feat1.shape[1], self.unified_channels, 1, bias=True)
        else:
            self.conv1 = nn.Identity()

        if feat2.shape[1] != self.unified_channels:
            self.conv2 = nn.Conv2d(feat2.shape[1], self.unified_channels, 1, bias=True)
        else:
            self.conv2 = nn.Identity()

        # 创建注意力模块
        self.spatial_attn = SpatialAttention()
        self.channel_attn = ChannelAttention(self.unified_channels, self.reduction)
        self.pixel_attn = PixelAttention(self.unified_channels)

        # 创建输出卷积（输出到c2指定的通道数）
        # 注意：使用self.original_c2作为目标通道数
        self.out_conv = nn.Conv2d(self.unified_channels, self.original_c2, 1, bias=True)

        # 移动所有模块到正确设备
        device = feat1.device
        dtype = feat1.dtype
        for module in [self.conv1, self.conv2, self.spatial_attn,
                      self.channel_attn, self.pixel_attn, self.out_conv]:
            if module is not None:
                module.to(device=device, dtype=dtype)

        # 初始化权重
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

        self._initialized = True

    def forward(self, x):
        # 输入检查
        if not isinstance(x, (list, tuple)):
            raise TypeError(f"CGAFusion需要列表/元组输入，得到 {type(x)}")

        if len(x) != 2:
            raise ValueError(f"CGAFusion需要2个特征图，得到 {len(x)}个")

        feat1, feat2 = x

        # 关键：只在第一次forward时初始化
        if not self._initialized:
            self._initialize_once(feat1, feat2)

        # 调试信息
        # print(f"CGAFusion前向传播: feat1={feat1.shape}, feat2={feat2.shape}")

        # 1. 调整通道数
        feat1_adjusted = self.conv1(feat1)
        feat2_adjusted = self.conv2(feat2)

        # 2. 调整空间尺寸（如果需要）
        if feat1_adjusted.shape[2:] != feat2_adjusted.shape[2:]:
            # 统一到较小的尺寸
            h1, w1 = feat1_adjusted.shape[2], feat1_adjusted.shape[3]
            h2, w2 = feat2_adjusted.shape[2], feat2_adjusted.shape[3]
            target_h, target_w = min(h1, h2), min(w1, w2)

            if (h1, w1) != (target_h, target_w):
                feat1_adjusted = F.interpolate(feat1_adjusted, size=(target_h, target_w),
                                              mode='bilinear', align_corners=False)

            if (h2, w2) != (target_h, target_w):
                feat2_adjusted = F.interpolate(feat2_adjusted, size=(target_h, target_w),
                                              mode='bilinear', align_corners=False)

        # 3. 初始加权融合
        initial = self.alpha * feat1_adjusted + (1 - self.alpha) * feat2_adjusted

        # 4. 计算注意力
        spatial_attn = self.spatial_attn(initial)
        spatial_attn = spatial_attn.expand_as(initial)

        channel_attn = self.channel_attn(initial)
        channel_attn = channel_attn.expand_as(initial)

        stage1_attn = spatial_attn + channel_attn

        # 5. 像素级注意力
        pixel_attn = self.pixel_attn(stage1_attn)

        # 6. 最终融合
        fused = initial + pixel_attn * feat1_adjusted + (1 - pixel_attn) * feat2_adjusted

        # 7. 输出调整
        output = self.out_conv(fused)

        # print(f"CGAFusion输出: {output.shape}")
        return output

Writing /content/ultralytics/ultralytics/nn/modules/CGAFusionFixed.py


- 修改模型的yaml

In [9]:
%%writefile  yolo_PCB.yaml
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLOv5 object detection model with P3/8 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolov5
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 6 # number of classes
scales: # model compound scaling constants, i.e. 'model=yolov5n.yaml' will call yolov5.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.33, 0.25, 1024]
  s: [0.33, 0.50, 1024]
  m: [0.67, 0.75, 1024]
  l: [1.00, 1.00, 1024]
  x: [1.33, 1.25, 1024]

# YOLOv5 v6.0 backbone
backbone:
  # [from, number, module, args]
  - [-1, 1, Conv, [64, 6, 2, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 3, C3FFTFixed, [128]]
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 6, C3, [256]]
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 9, C3, [512]]
  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 3, C3A2, [1024]]
  - [-1, 1, SPPF, [1024, 5]] # 9

# YOLOv5 v6.0 head
head:
  - [-1, 1, Conv, [512, 1, 1]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, CGAFusionFixed, [512]] # cat backbone P4，输出通道512
  - [-1, 3, C3, [512, False]] # 13

  - [-1, 1, Conv, [256, 1, 1]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, CGAFusionFixed, [256]] # cat backbone P3，输出通道256
  - [-1, 3, C3, [256, False]] # 17 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 14], 1, CGAFusionFixed, [512]] # cat head P4，输出通道512
  - [-1, 3, C3, [512, False]] # 20 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, CGAFusionFixed, [1024]] # cat head P5，输出通道1024
  - [-1, 3, C3, [1024, False]] # 23 (P5/32-large)

  - [[17, 20, 23], 1, Detect, [nc]] # Detect(P3, P4, P5)

hyp: /content/ultralytics/ultralytics/cfg/hyp_PCB.yaml

Writing yolo_PCB.yaml


- tasks.py:

In [ ]:
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

import contextlib
import pickle
import re
import types
from copy import deepcopy
from pathlib import Path

import torch
import torch.nn as nn

from ultralytics.nn.modules.C3FFTFixed import C3FFTFixed
from ultralytics.nn.modules.C3A2 import C3A2
from ultralytics.nn.modules.CGAFusionFixed import CGAFusionFixed
from ultralytics.nn.autobackend import check_class_names
from ultralytics.nn.modules import (
    AIFI,
    C1,
    C2,
    C2PSA,
    C3,
    C3TR,
    ELAN1,
    OBB,
    OBB26,
    PSA,
    SPP,
    SPPELAN,
    SPPF,
    A2C2f,
    AConv,
    ADown,
    Bottleneck,
    BottleneckCSP,
    C2f,
    C2fAttn,
    C2fCIB,
    C2fPSA,
    C3Ghost,
    C3k2,
    C3x,
    CBFuse,
    CBLinear,
    Classify,
    Concat,
    Conv,
    Conv2,
    ConvTranspose,
    Detect,
    DWConv,
    DWConvTranspose2d,
    Focus,
    GhostBottleneck,
    GhostConv,
    HGBlock,
    HGStem,
    ImagePoolingAttn,
    Index,
    LRPCHead,
    Pose,
    Pose26,
    RepC3,
    RepConv,
    RepNCSPELAN4,
    RepVGGDW,
    ResNetLayer,
    RTDETRDecoder,
    SCDown,
    Segment,
    Segment26,
    TorchVision,
    WorldDetect,
    YOLOEDetect,
    YOLOESegment,
    YOLOESegment26,
    v10Detect,
)
from ultralytics.utils import DEFAULT_CFG_DICT, LOGGER, YAML, colorstr, emojis
from ultralytics.utils.checks import check_requirements, check_suffix, check_yaml
from ultralytics.utils.loss import (
    E2EDetectLoss,
    v8ClassificationLoss,
    v8DetectionLoss,
    v8OBBLoss,
    v8PoseLoss,
    v8SegmentationLoss,
)
from ultralytics.utils.ops import make_divisible
from ultralytics.utils.patches import torch_load
from ultralytics.utils.plotting import feature_visualization
from ultralytics.utils.torch_utils import (
    fuse_conv_and_bn,
    fuse_deconv_and_bn,
    initialize_weights,
    intersect_dicts,
    model_info,
    scale_img,
    smart_inference_mode,
    time_sync,
)


class BaseModel(torch.nn.Module):
    """Base class for all YOLO models in the Ultralytics family.

    This class provides common functionality for YOLO models including forward pass handling, model fusion, information
    display, and weight loading capabilities.

    Attributes:
        model (torch.nn.Module): The neural network model.
        save (list): List of layer indices to save outputs from.
        stride (torch.Tensor): Model stride values.

    Methods:
        forward: Perform forward pass for training or inference.
        predict: Perform inference on input tensor.
        fuse: Fuse Conv2d and BatchNorm2d layers for optimization.
        info: Print model information.
        load: Load weights into the model.
        loss: Compute loss for training.

    Examples:
        Create a BaseModel instance
        >>> model = BaseModel()
        >>> model.info()  # Display model information
    """

    def forward(self, x, *args, **kwargs):
        """Perform forward pass of the model for either training or inference.

        If x is a dict, calculates and returns the loss for training. Otherwise, returns predictions for inference.

        Args:
            x (torch.Tensor | dict): Input tensor for inference, or dict with image tensor and labels for training.
            *args (Any): Variable length argument list.
            **kwargs (Any): Arbitrary keyword arguments.

        Returns:
            (torch.Tensor): Loss if x is a dict (training), or network predictions (inference).
        """
        if isinstance(x, dict):  # for cases of training and validating while training.
            return self.loss(x, *args, **kwargs)
        return self.predict(x, *args, **kwargs)

    def predict(self, x, profile=False, visualize=False, augment=False, embed=None):
        """Perform a forward pass through the network.

        Args:
            x (torch.Tensor): The input tensor to the model.
            profile (bool): Print the computation time of each layer if True.
            visualize (bool): Save the feature maps of the model if True.
            augment (bool): Augment image during prediction.
            embed (list, optional): A list of feature vectors/embeddings to return.

        Returns:
            (torch.Tensor): The last output of the model.
        """
        if augment:
            return self._predict_augment(x)
        return self._predict_once(x, profile, visualize, embed)

    def _predict_once(self, x, profile=False, visualize=False, embed=None):
        """Perform a forward pass through the network.

        Args:
            x (torch.Tensor): The input tensor to the model.
            profile (bool): Print the computation time of each layer if True.
            visualize (bool): Save the feature maps of the model if True.
            embed (list, optional): A list of feature vectors/embeddings to return.

        Returns:
            (torch.Tensor): The last output of the model.
        """
        y, dt, embeddings = [], [], []  # outputs
        embed = frozenset(embed) if embed is not None else {-1}
        max_idx = max(embed)
        for m in self.model:
            if m.f != -1:  # if not from previous layer
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]  # from earlier layers
            if profile:
                self._profile_one_layer(m, x, dt)
            x = m(x)  # run
            y.append(x if m.i in self.save else None)  # save output
            if visualize:
                feature_visualization(x, m.type, m.i, save_dir=visualize)
            if m.i in embed:
                embeddings.append(torch.nn.functional.adaptive_avg_pool2d(x, (1, 1)).squeeze(-1).squeeze(-1))  # flatten
                if m.i == max_idx:
                    return torch.unbind(torch.cat(embeddings, 1), dim=0)
        return x

    def _predict_augment(self, x):
        """Perform augmentations on input image x and return augmented inference."""
        LOGGER.warning(
            f"{self.__class__.__name__} does not support 'augment=True' prediction. "
            f"Reverting to single-scale prediction."
        )
        return self._predict_once(x)

    def _profile_one_layer(self, m, x, dt):
        """Profile the computation time and FLOPs of a single layer of the model on a given input.

        Args:
            m (torch.nn.Module): The layer to be profiled.
            x (torch.Tensor): The input data to the layer.
            dt (list): A list to store the computation time of the layer.
        """
        try:
            import thop
        except ImportError:
            thop = None  # conda support without 'ultralytics-thop' installed

        c = m == self.model[-1] and isinstance(x, list)  # is final layer list, copy input as inplace fix
        flops = thop.profile(m, inputs=[x.copy() if c else x], verbose=False)[0] / 1e9 * 2 if thop else 0  # GFLOPs
        t = time_sync()
        for _ in range(10):
            m(x.copy() if c else x)
        dt.append((time_sync() - t) * 100)
        if m == self.model[0]:
            LOGGER.info(f"{'time (ms)':>10s} {'GFLOPs':>10s} {'params':>10s}  module")
        LOGGER.info(f"{dt[-1]:10.2f} {flops:10.2f} {m.np:10.0f}  {m.type}")
        if c:
            LOGGER.info(f"{sum(dt):10.2f} {'-':>10s} {'-':>10s}  Total")

    def fuse(self, verbose=True):
        """Fuse the `Conv2d()` and `BatchNorm2d()` layers of the model into a single layer for improved computation
        efficiency.

        Returns:
            (torch.nn.Module): The fused model is returned.
        """
        if not self.is_fused():
            for m in self.model.modules():
                if isinstance(m, (Conv, Conv2, DWConv)) and hasattr(m, "bn"):
                    if isinstance(m, Conv2):
                        m.fuse_convs()
                    m.conv = fuse_conv_and_bn(m.conv, m.bn)  # update conv
                    delattr(m, "bn")  # remove batchnorm
                    m.forward = m.forward_fuse  # update forward
                if isinstance(m, ConvTranspose) and hasattr(m, "bn"):
                    m.conv_transpose = fuse_deconv_and_bn(m.conv_transpose, m.bn)
                    delattr(m, "bn")  # remove batchnorm
                    m.forward = m.forward_fuse  # update forward
                if isinstance(m, RepConv):
                    m.fuse_convs()
                    m.forward = m.forward_fuse  # update forward
                if isinstance(m, RepVGGDW):
                    m.fuse()
                    m.forward = m.forward_fuse
                if isinstance(m, Detect) and getattr(m, "end2end", False):
                    m.fuse()  # remove one2many head
            self.info(verbose=verbose)

        return self

    def is_fused(self, thresh=10):
        """Check if the model has less than a certain threshold of BatchNorm layers.

        Args:
            thresh (int, optional): The threshold number of BatchNorm layers.

        Returns:
            (bool): True if the number of BatchNorm layers in the model is less than the threshold, False otherwise.
        """
        bn = tuple(v for k, v in torch.nn.__dict__.items() if "Norm" in k)  # normalization layers, i.e. BatchNorm2d()
        return sum(isinstance(v, bn) for v in self.modules()) < thresh  # True if < 'thresh' BatchNorm layers in model

    def info(self, detailed=False, verbose=True, imgsz=640):
        """Print model information.

        Args:
            detailed (bool): If True, prints out detailed information about the model.
            verbose (bool): If True, prints out the model information.
            imgsz (int): The size of the image that the model will be trained on.
        """
        return model_info(self, detailed=detailed, verbose=verbose, imgsz=imgsz)

    def _apply(self, fn):
        """Apply a function to all tensors in the model that are not parameters or registered buffers.

        Args:
            fn (function): The function to apply to the model.

        Returns:
            (BaseModel): An updated BaseModel object.
        """
        self = super()._apply(fn)
        m = self.model[-1]  # Detect()
        if isinstance(
            m, Detect
        ):  # includes all Detect subclasses like Segment, Pose, OBB, WorldDetect, YOLOEDetect, YOLOESegment
            m.stride = fn(m.stride)
            m.anchors = fn(m.anchors)
            m.strides = fn(m.strides)
        return self

    def load(self, weights, verbose=True):
        """Load weights into the model.

        Args:
            weights (dict | torch.nn.Module): The pre-trained weights to be loaded.
            verbose (bool, optional): Whether to log the transfer progress.
        """
        model = weights["model"] if isinstance(weights, dict) else weights  # torchvision models are not dicts
        csd = model.float().state_dict()  # checkpoint state_dict as FP32
        updated_csd = intersect_dicts(csd, self.state_dict())  # intersect
        self.load_state_dict(updated_csd, strict=False)  # load
        len_updated_csd = len(updated_csd)
        first_conv = "model.0.conv.weight"  # hard-coded to yolo models for now
        # mostly used to boost multi-channel training
        state_dict = self.state_dict()
        if first_conv not in updated_csd and first_conv in state_dict:
            c1, c2, h, w = state_dict[first_conv].shape
            cc1, cc2, ch, cw = csd[first_conv].shape
            if ch == h and cw == w:
                c1, c2 = min(c1, cc1), min(c2, cc2)
                state_dict[first_conv][:c1, :c2] = csd[first_conv][:c1, :c2]
                len_updated_csd += 1
        if verbose:
            LOGGER.info(f"Transferred {len_updated_csd}/{len(self.model.state_dict())} items from pretrained weights")

    def loss(self, batch, preds=None):
        """Compute loss.

        Args:
            batch (dict): Batch to compute loss on.
            preds (torch.Tensor | list[torch.Tensor], optional): Predictions.
        """
        if getattr(self, "criterion", None) is None:
            self.criterion = self.init_criterion()

        if preds is None:
            preds = self.forward(batch["img"])
        return self.criterion(preds, batch)

    def init_criterion(self):
        """Initialize the loss criterion for the BaseModel."""
        raise NotImplementedError("compute_loss() needs to be implemented by task heads")


class DetectionModel(BaseModel):
    """YOLO detection model.

    This class implements the YOLO detection architecture, handling model initialization, forward pass, augmented
    inference, and loss computation for object detection tasks.

    Attributes:
        yaml (dict): Model configuration dictionary.
        model (torch.nn.Sequential): The neural network model.
        save (list): List of layer indices to save outputs from.
        names (dict): Class names dictionary.
        inplace (bool): Whether to use inplace operations.
        end2end (bool): Whether the model uses end-to-end detection.
        stride (torch.Tensor): Model stride values.

    Methods:
        __init__: Initialize the YOLO detection model.
        _predict_augment: Perform augmented inference.
        _descale_pred: De-scale predictions following augmented inference.
        _clip_augmented: Clip YOLO augmented inference tails.
        init_criterion: Initialize the loss criterion.

    Examples:
        Initialize a detection model
        >>> model = DetectionModel("yolo26n.yaml", ch=3, nc=80)
        >>> results = model.predict(image_tensor)
    """

    def __init__(self, cfg="yolo26n.yaml", ch=3, nc=None, verbose=True):
        """Initialize the YOLO detection model with the given config and parameters.

        Args:
            cfg (str | dict): Model configuration file path or dictionary.
            ch (int): Number of input channels.
            nc (int, optional): Number of classes.
            verbose (bool): Whether to display model information.
        """
        super().__init__()
        self.yaml = cfg if isinstance(cfg, dict) else yaml_model_load(cfg)  # cfg dict
        if self.yaml["backbone"][0][2] == "Silence":
            LOGGER.warning(
                "YOLOv9 `Silence` module is deprecated in favor of torch.nn.Identity. "
                "Please delete local *.pt file and re-download the latest model checkpoint."
            )
            self.yaml["backbone"][0][2] = "nn.Identity"

        # Define model
        self.yaml["channels"] = ch  # save channels
        if nc and nc != self.yaml["nc"]:
            LOGGER.info(f"Overriding model.yaml nc={self.yaml['nc']} with nc={nc}")
            self.yaml["nc"] = nc  # override YAML value
        self.model, self.save = parse_model(deepcopy(self.yaml), ch=ch, verbose=verbose)  # model, savelist
        self.names = {i: f"{i}" for i in range(self.yaml["nc"])}  # default names dict
        self.inplace = self.yaml.get("inplace", True)

        # Build strides
        m = self.model[-1]  # Detect()
        if isinstance(m, Detect):  # includes all Detect subclasses like Segment, Pose, OBB, YOLOEDetect, YOLOESegment
            s = 256  # 2x min stride
            m.inplace = self.inplace

            def _forward(x):
                """Perform a forward pass through the model, handling different Detect subclass types accordingly."""
                output = self.forward(x)
                if self.end2end:
                    output = output["one2many"]
                return output["feats"]

            self.model.eval()  # Avoid changing batch statistics until training begins
            m.training = True  # Setting it to True to properly return strides
            m.stride = torch.tensor([s / x.shape[-2] for x in _forward(torch.zeros(1, ch, s, s))])  # forward
            self.stride = m.stride
            self.model.train()  # Set model back to training(default) mode
            m.bias_init()  # only run once
        else:
            self.stride = torch.Tensor([32])  # default stride, e.g., RTDETR

        # Init weights, biases
        initialize_weights(self)
        if verbose:
            self.info()
            LOGGER.info("")

    @property
    def end2end(self):
        """Return whether the model uses end-to-end NMS-free detection."""
        return getattr(self.model[-1], "end2end", False)

    def _predict_augment(self, x):
        """Perform augmentations on input image x and return augmented inference and train outputs.

        Args:
            x (torch.Tensor): Input image tensor.

        Returns:
            (torch.Tensor): Augmented inference output.
        """
        if getattr(self, "end2end", False) or self.__class__.__name__ != "DetectionModel":
            LOGGER.warning("Model does not support 'augment=True', reverting to single-scale prediction.")
            return self._predict_once(x)
        img_size = x.shape[-2:]  # height, width
        s = [1, 0.83, 0.67]  # scales
        f = [None, 3, None]  # flips (2-ud, 3-lr)
        y = []  # outputs
        for si, fi in zip(s, f):
            xi = scale_img(x.flip(fi) if fi else x, si, gs=int(self.stride.max()))
            yi = super().predict(xi)[0]  # forward
            yi = self._descale_pred(yi, fi, si, img_size)
            y.append(yi)
        y = self._clip_augmented(y)  # clip augmented tails
        return torch.cat(y, -1), None  # augmented inference, train

    @staticmethod
    def _descale_pred(p, flips, scale, img_size, dim=1):
        """De-scale predictions following augmented inference (inverse operation).

        Args:
            p (torch.Tensor): Predictions tensor.
            flips (int): Flip type (0=none, 2=ud, 3=lr).
            scale (float): Scale factor.
            img_size (tuple): Original image size (height, width).
            dim (int): Dimension to split at.

        Returns:
            (torch.Tensor): De-scaled predictions.
        """
        p[:, :4] /= scale  # de-scale
        x, y, wh, cls = p.split((1, 1, 2, p.shape[dim] - 4), dim)
        if flips == 2:
            y = img_size[0] - y  # de-flip ud
        elif flips == 3:
            x = img_size[1] - x  # de-flip lr
        return torch.cat((x, y, wh, cls), dim)

    def _clip_augmented(self, y):
        """Clip YOLO augmented inference tails.

        Args:
            y (list[torch.Tensor]): List of detection tensors.

        Returns:
            (list[torch.Tensor]): Clipped detection tensors.
        """
        nl = self.model[-1].nl  # number of detection layers (P3-P5)
        g = sum(4**x for x in range(nl))  # grid points
        e = 1  # exclude layer count
        i = (y[0].shape[-1] // g) * sum(4**x for x in range(e))  # indices
        y[0] = y[0][..., :-i]  # large
        i = (y[-1].shape[-1] // g) * sum(4 ** (nl - 1 - x) for x in range(e))  # indices
        y[-1] = y[-1][..., i:]  # small
        return y

    def init_criterion(self):
        """Initialize the loss criterion for the DetectionModel."""
        return E2ELoss(self) if getattr(self, "end2end", False) else v8DetectionLoss(self)


class OBBModel(DetectionModel):
    """YOLO Oriented Bounding Box (OBB) model.

    This class extends DetectionModel to handle oriented bounding box detection tasks, providing specialized loss
    computation for rotated object detection.

    Methods:
        __init__: Initialize YOLO OBB model.
        init_criterion: Initialize the loss criterion for OBB detection.

    Examples:
        Initialize an OBB model
        >>> model = OBBModel("yolo26n-obb.yaml", ch=3, nc=80)
        >>> results = model.predict(image_tensor)
    """

    def __init__(self, cfg="yolo26n-obb.yaml", ch=3, nc=None, verbose=True):
        """Initialize YOLO OBB model with given config and parameters.

        Args:
            cfg (str | dict): Model configuration file path or dictionary.
            ch (int): Number of input channels.
            nc (int, optional): Number of classes.
            verbose (bool): Whether to display model information.
        """
        super().__init__(cfg=cfg, ch=ch, nc=nc, verbose=verbose)

    def init_criterion(self):
        """Initialize the loss criterion for the model."""
        return E2ELoss(self, v8OBBLoss) if getattr(self, "end2end", False) else v8OBBLoss(self)


class SegmentationModel(DetectionModel):
    """YOLO segmentation model.

    This class extends DetectionModel to handle instance segmentation tasks, providing specialized loss computation for
    pixel-level object detection and segmentation.

    Methods:
        __init__: Initialize YOLO segmentation model.
        init_criterion: Initialize the loss criterion for segmentation.

    Examples:
        Initialize a segmentation model
        >>> model = SegmentationModel("yolo26n-seg.yaml", ch=3, nc=80)
        >>> results = model.predict(image_tensor)
    """

    def __init__(self, cfg="yolo26n-seg.yaml", ch=3, nc=None, verbose=True):
        """Initialize Ultralytics YOLO segmentation model with given config and parameters.

        Args:
            cfg (str | dict): Model configuration file path or dictionary.
            ch (int): Number of input channels.
            nc (int, optional): Number of classes.
            verbose (bool): Whether to display model information.
        """
        super().__init__(cfg=cfg, ch=ch, nc=nc, verbose=verbose)

    def init_criterion(self):
        """Initialize the loss criterion for the SegmentationModel."""
        return E2ELoss(self, v8SegmentationLoss) if getattr(self, "end2end", False) else v8SegmentationLoss(self)


class PoseModel(DetectionModel):
    """YOLO pose model.

    This class extends DetectionModel to handle human pose estimation tasks, providing specialized loss computation for
    keypoint detection and pose estimation.

    Attributes:
        kpt_shape (tuple): Shape of keypoints data (num_keypoints, num_dimensions).

    Methods:
        __init__: Initialize YOLO pose model.
        init_criterion: Initialize the loss criterion for pose estimation.

    Examples:
        Initialize a pose model
        >>> model = PoseModel("yolo26n-pose.yaml", ch=3, nc=1, data_kpt_shape=(17, 3))
        >>> results = model.predict(image_tensor)
    """

    def __init__(self, cfg="yolo26n-pose.yaml", ch=3, nc=None, data_kpt_shape=(None, None), verbose=True):
        """Initialize Ultralytics YOLO Pose model.

        Args:
            cfg (str | dict): Model configuration file path or dictionary.
            ch (int): Number of input channels.
            nc (int, optional): Number of classes.
            data_kpt_shape (tuple): Shape of keypoints data.
            verbose (bool): Whether to display model information.
        """
        if not isinstance(cfg, dict):
            cfg = yaml_model_load(cfg)  # load model YAML
        if any(data_kpt_shape) and list(data_kpt_shape) != list(cfg["kpt_shape"]):
            LOGGER.info(f"Overriding model.yaml kpt_shape={cfg['kpt_shape']} with kpt_shape={data_kpt_shape}")
            cfg["kpt_shape"] = data_kpt_shape
        super().__init__(cfg=cfg, ch=ch, nc=nc, verbose=verbose)

    def init_criterion(self):
        """Initialize the loss criterion for the PoseModel."""
        return E2ELoss(self, PoseLoss26) if getattr(self, "end2end", False) else v8PoseLoss(self)


class ClassificationModel(BaseModel):
    """YOLO classification model.

    This class implements the YOLO classification architecture for image classification tasks, providing model
    initialization, configuration, and output reshaping capabilities.

    Attributes:
        yaml (dict): Model configuration dictionary.
        model (torch.nn.Sequential): The neural network model.
        stride (torch.Tensor): Model stride values.
        names (dict): Class names dictionary.

    Methods:
        __init__: Initialize ClassificationModel.
        _from_yaml: Set model configurations and define architecture.
        reshape_outputs: Update model to specified class count.
        init_criterion: Initialize the loss criterion.

    Examples:
        Initialize a classification model
        >>> model = ClassificationModel("yolo26n-cls.yaml", ch=3, nc=1000)
        >>> results = model.predict(image_tensor)
    """

    def __init__(self, cfg="yolo26n-cls.yaml", ch=3, nc=None, verbose=True):
        """Initialize ClassificationModel with YAML, channels, number of classes, verbose flag.

        Args:
            cfg (str | dict): Model configuration file path or dictionary.
            ch (int): Number of input channels.
            nc (int, optional): Number of classes.
            verbose (bool): Whether to display model information.
        """
        super().__init__()
        self._from_yaml(cfg, ch, nc, verbose)

    def _from_yaml(self, cfg, ch, nc, verbose):
        """Set Ultralytics YOLO model configurations and define the model architecture.

        Args:
            cfg (str | dict): Model configuration file path or dictionary.
            ch (int): Number of input channels.
            nc (int, optional): Number of classes.
            verbose (bool): Whether to display model information.
        """
        self.yaml = cfg if isinstance(cfg, dict) else yaml_model_load(cfg)  # cfg dict

        # Define model
        ch = self.yaml["channels"] = self.yaml.get("channels", ch)  # input channels
        if nc and nc != self.yaml["nc"]:
            LOGGER.info(f"Overriding model.yaml nc={self.yaml['nc']} with nc={nc}")
            self.yaml["nc"] = nc  # override YAML value
        elif not nc and not self.yaml.get("nc", None):
            raise ValueError("nc not specified. Must specify nc in model.yaml or function arguments.")
        self.model, self.save = parse_model(deepcopy(self.yaml), ch=ch, verbose=verbose)  # model, savelist
        self.stride = torch.Tensor([1])  # no stride constraints
        self.names = {i: f"{i}" for i in range(self.yaml["nc"])}  # default names dict
        self.info()

    @staticmethod
    def reshape_outputs(model, nc):
        """Update a TorchVision classification model to class count 'n' if required.

        Args:
            model (torch.nn.Module): Model to update.
            nc (int): New number of classes.
        """
        name, m = list((model.model if hasattr(model, "model") else model).named_children())[-1]  # last module
        if isinstance(m, Classify):  # YOLO Classify() head
            if m.linear.out_features != nc:
                m.linear = torch.nn.Linear(m.linear.in_features, nc)
        elif isinstance(m, torch.nn.Linear):  # ResNet, EfficientNet
            if m.out_features != nc:
                setattr(model, name, torch.nn.Linear(m.in_features, nc))
        elif isinstance(m, torch.nn.Sequential):
            types = [type(x) for x in m]
            if torch.nn.Linear in types:
                i = len(types) - 1 - types[::-1].index(torch.nn.Linear)  # last torch.nn.Linear index
                if m[i].out_features != nc:
                    m[i] = torch.nn.Linear(m[i].in_features, nc)
            elif torch.nn.Conv2d in types:
                i = len(types) - 1 - types[::-1].index(torch.nn.Conv2d)  # last torch.nn.Conv2d index
                if m[i].out_channels != nc:
                    m[i] = torch.nn.Conv2d(
                        m[i].in_channels, nc, m[i].kernel_size, m[i].stride, bias=m[i].bias is not None
                    )

    def init_criterion(self):
        """Initialize the loss criterion for the ClassificationModel."""
        return v8ClassificationLoss()


class RTDETRDetectionModel(DetectionModel):
    """RTDETR (Real-time DEtection and Tracking using Transformers) Detection Model class.

    This class is responsible for constructing the RTDETR architecture, defining loss functions, and facilitating both
    the training and inference processes. RTDETR is an object detection and tracking model that extends from the
    DetectionModel base class.

    Attributes:
        nc (int): Number of classes for detection.
        criterion (RTDETRDetectionLoss): Loss function for training.

    Methods:
        __init__: Initialize the RTDETRDetectionModel.
        init_criterion: Initialize the loss criterion.
        loss: Compute loss for training.
        predict: Perform forward pass through the model.

    Examples:
        Initialize an RTDETR model
        >>> model = RTDETRDetectionModel("rtdetr-l.yaml", ch=3, nc=80)
        >>> results = model.predict(image_tensor)
    """

    def __init__(self, cfg="rtdetr-l.yaml", ch=3, nc=None, verbose=True):
        """Initialize the RTDETRDetectionModel.

        Args:
            cfg (str | dict): Configuration file name or path.
            ch (int): Number of input channels.
            nc (int, optional): Number of classes.
            verbose (bool): Print additional information during initialization.
        """
        super().__init__(cfg=cfg, ch=ch, nc=nc, verbose=verbose)

    def _apply(self, fn):
        """Apply a function to all tensors in the model that are not parameters or registered buffers.

        Args:
            fn (function): The function to apply to the model.

        Returns:
            (RTDETRDetectionModel): An updated BaseModel object.
        """
        self = super()._apply(fn)
        m = self.model[-1]
        m.anchors = fn(m.anchors)
        m.valid_mask = fn(m.valid_mask)
        return self

    def init_criterion(self):
        """Initialize the loss criterion for the RTDETRDetectionModel."""
        from ultralytics.models.utils.loss import RTDETRDetectionLoss

        return RTDETRDetectionLoss(nc=self.nc, use_vfl=True)

    def loss(self, batch, preds=None):
        """Compute the loss for the given batch of data.

        Args:
            batch (dict): Dictionary containing image and label data.
            preds (torch.Tensor, optional): Precomputed model predictions.

        Returns:
            loss_sum (torch.Tensor): Total loss value.
            loss_items (torch.Tensor): Main three losses in a tensor.
        """
        if not hasattr(self, "criterion"):
            self.criterion = self.init_criterion()

        img = batch["img"]
        # NOTE: preprocess gt_bbox and gt_labels to list.
        bs = img.shape[0]
        batch_idx = batch["batch_idx"]
        gt_groups = [(batch_idx == i).sum().item() for i in range(bs)]
        targets = {
            "cls": batch["cls"].to(img.device, dtype=torch.long).view(-1),
            "bboxes": batch["bboxes"].to(device=img.device),
            "batch_idx": batch_idx.to(img.device, dtype=torch.long).view(-1),
            "gt_groups": gt_groups,
        }

        if preds is None:
            preds = self.predict(img, batch=targets)
        dec_bboxes, dec_scores, enc_bboxes, enc_scores, dn_meta = preds if self.training else preds[1]
        if dn_meta is None:
            dn_bboxes, dn_scores = None, None
        else:
            dn_bboxes, dec_bboxes = torch.split(dec_bboxes, dn_meta["dn_num_split"], dim=2)
            dn_scores, dec_scores = torch.split(dec_scores, dn_meta["dn_num_split"], dim=2)

        dec_bboxes = torch.cat([enc_bboxes.unsqueeze(0), dec_bboxes])  # (7, bs, 300, 4)
        dec_scores = torch.cat([enc_scores.unsqueeze(0), dec_scores])

        loss = self.criterion(
            (dec_bboxes, dec_scores), targets, dn_bboxes=dn_bboxes, dn_scores=dn_scores, dn_meta=dn_meta
        )
        # NOTE: There are like 12 losses in RTDETR, backward with all losses but only show the main three losses.
        return sum(loss.values()), torch.as_tensor(
            [loss[k].detach() for k in ["loss_giou", "loss_class", "loss_bbox"]], device=img.device
        )

    def predict(self, x, profile=False, visualize=False, batch=None, augment=False, embed=None):
        """Perform a forward pass through the model.

        Args:
            x (torch.Tensor): The input tensor.
            profile (bool): If True, profile the computation time for each layer.
            visualize (bool): If True, save feature maps for visualization.
            batch (dict, optional): Ground truth data for evaluation.
            augment (bool): If True, perform data augmentation during inference.
            embed (list, optional): A list of feature vectors/embeddings to return.

        Returns:
            (torch.Tensor): Model's output tensor.
        """
        y, dt, embeddings = [], [], []  # outputs
        embed = frozenset(embed) if embed is not None else {-1}
        max_idx = max(embed)
        for m in self.model[:-1]:  # except the head part
            if m.f != -1:  # if not from previous layer
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]  # from earlier layers
            if profile:
                self._profile_one_layer(m, x, dt)
            x = m(x)  # run
            y.append(x if m.i in self.save else None)  # save output
            if visualize:
                feature_visualization(x, m.type, m.i, save_dir=visualize)
            if m.i in embed:
                embeddings.append(torch.nn.functional.adaptive_avg_pool2d(x, (1, 1)).squeeze(-1).squeeze(-1))  # flatten
                if m.i == max_idx:
                    return torch.unbind(torch.cat(embeddings, 1), dim=0)
        head = self.model[-1]
        x = head([y[j] for j in head.f], batch)  # head inference
        return x


class WorldModel(DetectionModel):
    """YOLOv8 World Model.

    This class implements the YOLOv8 World model for open-vocabulary object detection, supporting text-based class
    specification and CLIP model integration for zero-shot detection capabilities.

    Attributes:
        txt_feats (torch.Tensor): Text feature embeddings for classes.
        clip_model (torch.nn.Module): CLIP model for text encoding.

    Methods:
        __init__: Initialize YOLOv8 world model.
        set_classes: Set classes for offline inference.
        get_text_pe: Get text positional embeddings.
        predict: Perform forward pass with text features.
        loss: Compute loss with text features.

    Examples:
        Initialize a world model
        >>> model = WorldModel("yolov8s-world.yaml", ch=3, nc=80)
        >>> model.set_classes(["person", "car", "bicycle"])
        >>> results = model.predict(image_tensor)
    """

    def __init__(self, cfg="yolov8s-world.yaml", ch=3, nc=None, verbose=True):
        """Initialize YOLOv8 world model with given config and parameters.

        Args:
            cfg (str | dict): Model configuration file path or dictionary.
            ch (int): Number of input channels.
            nc (int, optional): Number of classes.
            verbose (bool): Whether to display model information.
        """
        self.txt_feats = torch.randn(1, nc or 80, 512)  # features placeholder
        self.clip_model = None  # CLIP model placeholder
        super().__init__(cfg=cfg, ch=ch, nc=nc, verbose=verbose)

    def set_classes(self, text, batch=80, cache_clip_model=True):
        """Set classes in advance so that model could do offline-inference without clip model.

        Args:
            text (list[str]): List of class names.
            batch (int): Batch size for processing text tokens.
            cache_clip_model (bool): Whether to cache the CLIP model.
        """
        self.txt_feats = self.get_text_pe(text, batch=batch, cache_clip_model=cache_clip_model)
        self.model[-1].nc = len(text)

    def get_text_pe(self, text, batch=80, cache_clip_model=True):
        """Get text positional embeddings for offline inference without CLIP model.

        Args:
            text (list[str]): List of class names.
            batch (int): Batch size for processing text tokens.
            cache_clip_model (bool): Whether to cache the CLIP model.

        Returns:
            (torch.Tensor): Text positional embeddings.
        """
        from ultralytics.nn.text_model import build_text_model

        device = next(self.model.parameters()).device
        if not getattr(self, "clip_model", None) and cache_clip_model:
            # For backwards compatibility of models lacking clip_model attribute
            self.clip_model = build_text_model("clip:ViT-B/32", device=device)
        model = self.clip_model if cache_clip_model else build_text_model("clip:ViT-B/32", device=device)
        text_token = model.tokenize(text)
        txt_feats = [model.encode_text(token).detach() for token in text_token.split(batch)]
        txt_feats = txt_feats[0] if len(txt_feats) == 1 else torch.cat(txt_feats, dim=0)
        return txt_feats.reshape(-1, len(text), txt_feats.shape[-1])

    def predict(self, x, profile=False, visualize=False, txt_feats=None, augment=False, embed=None):
        """Perform a forward pass through the model.

        Args:
            x (torch.Tensor): The input tensor.
            profile (bool): If True, profile the computation time for each layer.
            visualize (bool): If True, save feature maps for visualization.
            txt_feats (torch.Tensor, optional): The text features, use it if it's given.
            augment (bool): If True, perform data augmentation during inference.
            embed (list, optional): A list of feature vectors/embeddings to return.

        Returns:
            (torch.Tensor): Model's output tensor.
        """
        txt_feats = (self.txt_feats if txt_feats is None else txt_feats).to(device=x.device, dtype=x.dtype)
        if txt_feats.shape[0] != x.shape[0] or self.model[-1].export:
            txt_feats = txt_feats.expand(x.shape[0], -1, -1)
        ori_txt_feats = txt_feats.clone()
        y, dt, embeddings = [], [], []  # outputs
        embed = frozenset(embed) if embed is not None else {-1}
        max_idx = max(embed)
        for m in self.model:  # except the head part
            if m.f != -1:  # if not from previous layer
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]  # from earlier layers
            if profile:
                self._profile_one_layer(m, x, dt)
            if isinstance(m, C2fAttn):
                x = m(x, txt_feats)
            elif isinstance(m, WorldDetect):
                x = m(x, ori_txt_feats)
            elif isinstance(m, ImagePoolingAttn):
                txt_feats = m(x, txt_feats)
            else:
                x = m(x)  # run

            y.append(x if m.i in self.save else None)  # save output
            if visualize:
                feature_visualization(x, m.type, m.i, save_dir=visualize)
            if m.i in embed:
                embeddings.append(torch.nn.functional.adaptive_avg_pool2d(x, (1, 1)).squeeze(-1).squeeze(-1))  # flatten
                if m.i == max_idx:
                    return torch.unbind(torch.cat(embeddings, 1), dim=0)
        return x

    def loss(self, batch, preds=None):
        """Compute loss.

        Args:
            batch (dict): Batch to compute loss on.
            preds (torch.Tensor | list[torch.Tensor], optional): Predictions.
        """
        if not hasattr(self, "criterion"):
            self.criterion = self.init_criterion()

        if preds is None:
            preds = self.forward(batch["img"], txt_feats=batch["txt_feats"])
        return self.criterion(preds, batch)


class YOLOEModel(DetectionModel):
    """YOLOE detection model.

    This class implements the YOLOE architecture for efficient object detection with text and visual prompts, supporting
    both prompt-based and prompt-free inference modes.

    Attributes:
        pe (torch.Tensor): Prompt embeddings for classes.
        clip_model (torch.nn.Module): CLIP model for text encoding.

    Methods:
        __init__: Initialize YOLOE model.
        get_text_pe: Get text positional embeddings.
        get_visual_pe: Get visual embeddings.
        set_vocab: Set vocabulary for prompt-free model.
        get_vocab: Get fused vocabulary layer.
        set_classes: Set classes for offline inference.
        get_cls_pe: Get class positional embeddings.
        predict: Perform forward pass with prompts.
        loss: Compute loss with prompts.

    Examples:
        Initialize a YOLOE model
        >>> model = YOLOEModel("yoloe-v8s.yaml", ch=3, nc=80)
        >>> results = model.predict(image_tensor, tpe=text_embeddings)
    """

    def __init__(self, cfg="yoloe-v8s.yaml", ch=3, nc=None, verbose=True):
        """Initialize YOLOE model with given config and parameters.

        Args:
            cfg (str | dict): Model configuration file path or dictionary.
            ch (int): Number of input channels.
            nc (int, optional): Number of classes.
            verbose (bool): Whether to display model information.
        """
        super().__init__(cfg=cfg, ch=ch, nc=nc, verbose=verbose)
        self.text_model = self.yaml.get("text_model", "mobileclip:blt")

    @smart_inference_mode()
    def get_text_pe(self, text, batch=80, cache_clip_model=False, without_reprta=False):
        """Get text positional embeddings for offline inference without CLIP model.

        Args:
            text (list[str]): List of class names.
            batch (int): Batch size for processing text tokens.
            cache_clip_model (bool): Whether to cache the CLIP model.
            without_reprta (bool): Whether to return text embeddings without reprta module processing.

        Returns:
            (torch.Tensor): Text positional embeddings.
        """
        from ultralytics.nn.text_model import build_text_model

        device = next(self.model.parameters()).device
        if not getattr(self, "clip_model", None) and cache_clip_model:
            # For backwards compatibility of models lacking clip_model attribute
            self.clip_model = build_text_model(getattr(self, "text_model", "mobileclip:blt"), device=device)

        model = (
            self.clip_model
            if cache_clip_model
            else build_text_model(getattr(self, "text_model", "mobileclip:blt"), device=device)
        )
        text_token = model.tokenize(text)
        txt_feats = [model.encode_text(token).detach() for token in text_token.split(batch)]
        txt_feats = txt_feats[0] if len(txt_feats) == 1 else torch.cat(txt_feats, dim=0)
        txt_feats = txt_feats.reshape(-1, len(text), txt_feats.shape[-1])
        if without_reprta:
            return txt_feats

        head = self.model[-1]
        assert isinstance(head, YOLOEDetect)
        return head.get_tpe(txt_feats)  # run auxiliary text head

    @smart_inference_mode()
    def get_visual_pe(self, img, visual):
        """Get visual embeddings.

        Args:
            img (torch.Tensor): Input image tensor.
            visual (torch.Tensor): Visual features.

        Returns:
            (torch.Tensor): Visual positional embeddings.
        """
        return self(img, vpe=visual, return_vpe=True)

    def set_vocab(self, vocab, names):
        """Set vocabulary for the prompt-free model.

        Args:
            vocab (nn.ModuleList): List of vocabulary items.
            names (list[str]): List of class names.
        """
        assert not self.training
        head = self.model[-1]
        assert isinstance(head, YOLOEDetect)

        # Cache anchors for head
        device = next(self.parameters()).device
        self(torch.empty(1, 3, self.args["imgsz"], self.args["imgsz"]).to(device))  # warmup

        cv3 = getattr(head, "one2one_cv3", head.cv3)
        cv2 = getattr(head, "one2one_cv2", head.cv2)

        # re-parameterization for prompt-free model
        self.model[-1].lrpc = nn.ModuleList(
            LRPCHead(cls, pf[-1], loc[-1], enabled=i != 2) for i, (cls, pf, loc) in enumerate(zip(vocab, cv3, cv2))
        )
        for loc_head, cls_head in zip(head.cv2, head.cv3):
            assert isinstance(loc_head, nn.Sequential)
            assert isinstance(cls_head, nn.Sequential)
            del loc_head[-1]
            del cls_head[-1]
        self.model[-1].nc = len(names)
        self.names = check_class_names(names)

    def get_vocab(self, names):
        """Get fused vocabulary layer from the model.

        Args:
            names (list): List of class names.

        Returns:
            (nn.ModuleList): List of vocabulary modules.
        """
        assert not self.training
        head = self.model[-1]
        assert isinstance(head, YOLOEDetect)
        assert not head.is_fused

        tpe = self.get_text_pe(names)
        self.set_classes(names, tpe)
        device = next(self.model.parameters()).device
        head.fuse(self.pe.to(device))  # fuse prompt embeddings to classify head

        cv3 = getattr(head, "one2one_cv3", head.cv3)
        vocab = nn.ModuleList()
        for cls_head in cv3:
            assert isinstance(cls_head, nn.Sequential)
            vocab.append(cls_head[-1])
        return vocab

    def set_classes(self, names, embeddings):
        """Set classes in advance so that model could do offline-inference without clip model.

        Args:
            names (list[str]): List of class names.
            embeddings (torch.Tensor): Embeddings tensor.
        """
        assert not hasattr(self.model[-1], "lrpc"), (
            "Prompt-free model does not support setting classes. Please try with Text/Visual prompt models."
        )
        assert embeddings.ndim == 3
        self.pe = embeddings
        self.model[-1].nc = len(names)
        self.names = check_class_names(names)

    def get_cls_pe(self, tpe, vpe):
        """Get class positional embeddings.

        Args:
            tpe (torch.Tensor, optional): Text positional embeddings.
            vpe (torch.Tensor, optional): Visual positional embeddings.

        Returns:
            (torch.Tensor): Class positional embeddings.
        """
        all_pe = []
        if tpe is not None:
            assert tpe.ndim == 3
            all_pe.append(tpe)
        if vpe is not None:
            assert vpe.ndim == 3
            all_pe.append(vpe)
        if not all_pe:
            all_pe.append(getattr(self, "pe", torch.zeros(1, 80, 512)))
        return torch.cat(all_pe, dim=1)

    def predict(
        self, x, profile=False, visualize=False, tpe=None, augment=False, embed=None, vpe=None, return_vpe=False
    ):
        """Perform a forward pass through the model.

        Args:
            x (torch.Tensor): The input tensor.
            profile (bool): If True, profile the computation time for each layer.
            visualize (bool): If True, save feature maps for visualization.
            tpe (torch.Tensor, optional): Text positional embeddings.
            augment (bool): If True, perform data augmentation during inference.
            embed (list, optional): A list of feature vectors/embeddings to return.
            vpe (torch.Tensor, optional): Visual positional embeddings.
            return_vpe (bool): If True, return visual positional embeddings.

        Returns:
            (torch.Tensor): Model's output tensor.
        """
        y, dt, embeddings = [], [], []  # outputs
        b = x.shape[0]
        embed = frozenset(embed) if embed is not None else {-1}
        max_idx = max(embed)
        for m in self.model:  # except the head part
            if m.f != -1:  # if not from previous layer
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]  # from earlier layers
            if profile:
                self._profile_one_layer(m, x, dt)
            if isinstance(m, YOLOEDetect):
                vpe = m.get_vpe(x, vpe) if vpe is not None else None
                if return_vpe:
                    assert vpe is not None
                    assert not self.training
                    return vpe
                cls_pe = self.get_cls_pe(m.get_tpe(tpe), vpe).to(device=x[0].device, dtype=x[0].dtype)
                if cls_pe.shape[0] != b or m.export:
                    cls_pe = cls_pe.expand(b, -1, -1)
                x.append(cls_pe)  # adding cls embedding
            x = m(x)  # run

            y.append(x if m.i in self.save else None)  # save output
            if visualize:
                feature_visualization(x, m.type, m.i, save_dir=visualize)
            if m.i in embed:
                embeddings.append(torch.nn.functional.adaptive_avg_pool2d(x, (1, 1)).squeeze(-1).squeeze(-1))  # flatten
                if m.i == max_idx:
                    return torch.unbind(torch.cat(embeddings, 1), dim=0)
        return x

    def loss(self, batch, preds=None):
        """Compute loss.

        Args:
            batch (dict): Batch to compute loss on.
            preds (torch.Tensor | list[torch.Tensor], optional): Predictions.
        """
        if not hasattr(self, "criterion"):
            from ultralytics.utils.loss import TVPDetectLoss

            visual_prompt = batch.get("visuals", None) is not None  # TODO
            self.criterion = (
                (E2ELoss(self, TVPDetectLoss) if getattr(self, "end2end", False) else TVPDetectLoss(self))
                if visual_prompt
                else self.init_criterion()
            )
        if preds is None:
            preds = self.forward(
                batch["img"],
                tpe=None if "visuals" in batch else batch.get("txt_feats", None),
                vpe=batch.get("visuals", None),
            )
        return self.criterion(preds, batch)


class YOLOESegModel(YOLOEModel, SegmentationModel):
    """YOLOE segmentation model.

    This class extends YOLOEModel to handle instance segmentation tasks with text and visual prompts, providing
    specialized loss computation for pixel-level object detection and segmentation.

    Methods:
        __init__: Initialize YOLOE segmentation model.
        loss: Compute loss with prompts for segmentation.

    Examples:
        Initialize a YOLOE segmentation model
        >>> model = YOLOESegModel("yoloe-v8s-seg.yaml", ch=3, nc=80)
        >>> results = model.predict(image_tensor, tpe=text_embeddings)
    """

    def __init__(self, cfg="yoloe-v8s-seg.yaml", ch=3, nc=None, verbose=True):
        """Initialize YOLOE segmentation model with given config and parameters.

        Args:
            cfg (str | dict): Model configuration file path or dictionary.
            ch (int): Number of input channels.
            nc (int, optional): Number of classes.
            verbose (bool): Whether to display model information.
        """
        super().__init__(cfg=cfg, ch=ch, nc=nc, verbose=verbose)

    def loss(self, batch, preds=None):
        """Compute loss.

        Args:
            batch (dict): Batch to compute loss on.
            preds (torch.Tensor | list[torch.Tensor], optional): Predictions.
        """
        if not hasattr(self, "criterion"):
            from ultralytics.utils.loss import TVPSegmentLoss

            visual_prompt = batch.get("visuals", None) is not None  # TODO
            self.criterion = (
                (E2ELoss(self, TVPSegmentLoss) if getattr(self, "end2end", False) else TVPSegmentLoss(self))
                if visual_prompt
                else self.init_criterion()
            )

        if preds is None:
            preds = self.forward(batch["img"], tpe=batch.get("txt_feats", None), vpe=batch.get("visuals", None))
        return self.criterion(preds, batch)


class Ensemble(torch.nn.ModuleList):
    """Ensemble of models.

    This class allows combining multiple YOLO models into an ensemble for improved performance through model averaging
    or other ensemble techniques.

    Methods:
        __init__: Initialize an ensemble of models.
        forward: Generate predictions from all models in the ensemble.

    Examples:
        Create an ensemble of models
        >>> ensemble = Ensemble()
        >>> ensemble.append(model1)
        >>> ensemble.append(model2)
        >>> results = ensemble(image_tensor)
    """

    def __init__(self):
        """Initialize an ensemble of models."""
        super().__init__()

    def forward(self, x, augment=False, profile=False, visualize=False):
        """Generate the YOLO network's final layer.

        Args:
            x (torch.Tensor): Input tensor.
            augment (bool): Whether to augment the input.
            profile (bool): Whether to profile the model.
            visualize (bool): Whether to visualize the features.

        Returns:
            y (torch.Tensor): Concatenated predictions from all models.
            train_out (None): Always None for ensemble inference.
        """
        y = [module(x, augment, profile, visualize)[0] for module in self]
        # y = torch.stack(y).max(0)[0]  # max ensemble
        # y = torch.stack(y).mean(0)  # mean ensemble
        y = torch.cat(y, 2)  # nms ensemble, y shape(B, HW, C*num_models)
        return y, None  # inference, train output


# Functions ------------------------------------------------------------------------------------------------------------


@contextlib.contextmanager
def temporary_modules(modules=None, attributes=None):
    """Context manager for temporarily adding or modifying modules in Python's module cache (`sys.modules`).

    This function can be used to change the module paths during runtime. It's useful when refactoring code, where you've
    moved a module from one location to another, but you still want to support the old import paths for backwards
    compatibility.

    Args:
        modules (dict, optional): A dictionary mapping old module paths to new module paths.
        attributes (dict, optional): A dictionary mapping old module attributes to new module attributes.

    Examples:
        >>> with temporary_modules({"old.module": "new.module"}, {"old.module.attribute": "new.module.attribute"}):
        >>> import old.module  # this will now import new.module
        >>> from old.module import attribute  # this will now import new.module.attribute

    Notes:
        The changes are only in effect inside the context manager and are undone once the context manager exits.
        Be aware that directly manipulating `sys.modules` can lead to unpredictable results, especially in larger
        applications or libraries. Use this function with caution.
    """
    if modules is None:
        modules = {}
    if attributes is None:
        attributes = {}
    import sys
    from importlib import import_module

    try:
        # Set attributes in sys.modules under their old name
        for old, new in attributes.items():
            old_module, old_attr = old.rsplit(".", 1)
            new_module, new_attr = new.rsplit(".", 1)
            setattr(import_module(old_module), old_attr, getattr(import_module(new_module), new_attr))

        # Set modules in sys.modules under their old name
        for old, new in modules.items():
            sys.modules[old] = import_module(new)

        yield
    finally:
        # Remove the temporary module paths
        for old in modules:
            if old in sys.modules:
                del sys.modules[old]


class SafeClass:
    """A placeholder class to replace unknown classes during unpickling."""

    def __init__(self, *args, **kwargs):
        """Initialize SafeClass instance, ignoring all arguments."""
        pass

    def __call__(self, *args, **kwargs):
        """Run SafeClass instance, ignoring all arguments."""
        pass


class SafeUnpickler(pickle.Unpickler):
    """Custom Unpickler that replaces unknown classes with SafeClass."""

    def find_class(self, module, name):
        """Attempt to find a class, returning SafeClass if not among safe modules.

        Args:
            module (str): Module name.
            name (str): Class name.

        Returns:
            (type): Found class or SafeClass.
        """
        safe_modules = (
            "torch",
            "collections",
            "collections.abc",
            "builtins",
            "math",
            "numpy",
            # Add other modules considered safe
        )
        if module in safe_modules:
            return super().find_class(module, name)
        else:
            return SafeClass


def torch_safe_load(weight, safe_only=False):
    """Attempt to load a PyTorch model with the torch.load() function. If a ModuleNotFoundError is raised, it catches
    the error, logs a warning message, and attempts to install the missing module via the check_requirements()
    function. After installation, the function again attempts to load the model using torch.load().

    Args:
        weight (str): The file path of the PyTorch model.
        safe_only (bool): If True, replace unknown classes with SafeClass during loading.

    Returns:
        ckpt (dict): The loaded model checkpoint.
        file (str): The loaded filename.

    Examples:
        >>> from ultralytics.nn.tasks import torch_safe_load
        >>> ckpt, file = torch_safe_load("path/to/best.pt", safe_only=True)
    """
    from ultralytics.utils.downloads import attempt_download_asset

    check_suffix(file=weight, suffix=".pt")
    file = attempt_download_asset(weight)  # search online if missing locally
    try:
        with temporary_modules(
            modules={
                "ultralytics.yolo.utils": "ultralytics.utils",
                "ultralytics.yolo.v8": "ultralytics.models.yolo",
                "ultralytics.yolo.data": "ultralytics.data",
            },
            attributes={
                "ultralytics.nn.modules.block.Silence": "torch.nn.Identity",  # YOLOv9e
                "ultralytics.nn.tasks.YOLOv10DetectionModel": "ultralytics.nn.tasks.DetectionModel",  # YOLOv10
                "ultralytics.utils.loss.v10DetectLoss": "ultralytics.utils.loss.E2EDetectLoss",  # YOLOv10
            },
        ):
            if safe_only:
                # Load via custom pickle module
                safe_pickle = types.ModuleType("safe_pickle")
                safe_pickle.Unpickler = SafeUnpickler
                safe_pickle.load = lambda file_obj: SafeUnpickler(file_obj).load()
                with open(file, "rb") as f:
                    ckpt = torch_load(f, pickle_module=safe_pickle)
            else:
                ckpt = torch_load(file, map_location="cpu")

    except ModuleNotFoundError as e:  # e.name is missing module name
        if e.name == "models":
            raise TypeError(
                emojis(
                    f"ERROR ❌️ {weight} appears to be an Ultralytics YOLOv5 model originally trained "
                    f"with https://github.com/ultralytics/yolov5.\nThis model is NOT forwards compatible with "
                    f"YOLOv8 at https://github.com/ultralytics/ultralytics."
                    f"\nRecommend fixes are to train a new model using the latest 'ultralytics' package or to "
                    f"run a command with an official Ultralytics model, i.e. 'yolo predict model=yolo26n.pt'"
                )
            ) from e
        elif e.name == "numpy._core":
            raise ModuleNotFoundError(
                emojis(
                    f"ERROR ❌️ {weight} requires numpy>=1.26.1, however numpy=={__import__('numpy').__version__} is installed."
                )
            ) from e
        LOGGER.warning(
            f"{weight} appears to require '{e.name}', which is not in Ultralytics requirements."
            f"\nAutoInstall will run now for '{e.name}' but this feature will be removed in the future."
            f"\nRecommend fixes are to train a new model using the latest 'ultralytics' package or to "
            f"run a command with an official Ultralytics model, i.e. 'yolo predict model=yolo26n.pt'"
        )
        check_requirements(e.name)  # install missing module
        ckpt = torch_load(file, map_location="cpu")

    if not isinstance(ckpt, dict):
        # File is likely a YOLO instance saved with i.e. torch.save(model, "saved_model.pt")
        LOGGER.warning(
            f"The file '{weight}' appears to be improperly saved or formatted. "
            f"For optimal results, use model.save('filename.pt') to correctly save YOLO models."
        )
        ckpt = {"model": ckpt.model}

    return ckpt, file


def load_checkpoint(weight, device=None, inplace=True, fuse=False):
    """Load a single model weights.

    Args:
        weight (str | Path): Model weight path.
        device (torch.device, optional): Device to load model to.
        inplace (bool): Whether to do inplace operations.
        fuse (bool): Whether to fuse model.

    Returns:
        model (torch.nn.Module): Loaded model.
        ckpt (dict): Model checkpoint dictionary.
    """
    ckpt, weight = torch_safe_load(weight)  # load ckpt
    args = {**DEFAULT_CFG_DICT, **(ckpt.get("train_args", {}))}  # combine model and default args, preferring model args
    model = (ckpt.get("ema") or ckpt["model"]).float()  # FP32 model

    # Model compatibility updates
    model.args = args  # attach args to model
    model.pt_path = weight  # attach *.pt file path to model
    model.task = getattr(model, "task", guess_model_task(model))
    if not hasattr(model, "stride"):
        model.stride = torch.tensor([32.0])

    model = (model.fuse() if fuse and hasattr(model, "fuse") else model).eval().to(device)  # model in eval mode

    # Module updates
    for m in model.modules():
        if hasattr(m, "inplace"):
            m.inplace = inplace
        elif isinstance(m, torch.nn.Upsample) and not hasattr(m, "recompute_scale_factor"):
            m.recompute_scale_factor = None  # torch 1.11.0 compatibility

    # Return model and ckpt
    return model, ckpt


def parse_model(d, ch, verbose=True):
    """Parse a YOLO model.yaml dictionary into a PyTorch model.

    Args:
        d (dict): Model dictionary.
        ch (int): Input channels.
        verbose (bool): Whether to print model details.

    Returns:
        model (torch.nn.Sequential): PyTorch model.
        save (list): Sorted list of output layers.
    """
    import ast

    # Args
    legacy = True  # backward compatibility for v3/v5/v8/v9 models
    max_channels = float("inf")
    nc, act, scales, end2end = (d.get(x) for x in ("nc", "activation", "scales", "end2end"))
    reg_max = d.get("reg_max", 16)
    depth, width, kpt_shape = (d.get(x, 1.0) for x in ("depth_multiple", "width_multiple", "kpt_shape"))
    scale = d.get("scale")
    if scales:
        if not scale:
            scale = next(iter(scales.keys()))
            LOGGER.warning(f"no model scale passed. Assuming scale='{scale}'.")
        depth, width, max_channels = scales[scale]

    if act:
        Conv.default_act = eval(act)  # redefine default activation, i.e. Conv.default_act = torch.nn.SiLU()
        if verbose:
            LOGGER.info(f"{colorstr('activation:')} {act}")  # print

    if verbose:
        LOGGER.info(f"\n{'':>3}{'from':>20}{'n':>3}{'params':>10}  {'module':<45}{'arguments':<30}")
    ch = [ch]
    layers, save, c2 = [], [], ch[-1]  # layers, savelist, ch out
    base_modules = frozenset(
        {
            Classify,
            Conv,
            ConvTranspose,
            GhostConv,
            Bottleneck,
            GhostBottleneck,
            SPP,
            SPPF,
            C2fPSA,
            C2PSA,
            DWConv,
            Focus,
            BottleneckCSP,
            C1,
            C2,
            C2f,
            C3k2,
            RepNCSPELAN4,
            ELAN1,
            ADown,
            AConv,
            SPPELAN,
            C2fAttn,
            C3,
            C3TR,
            C3Ghost,
            torch.nn.ConvTranspose2d,
            DWConvTranspose2d,
            C3x,
            RepC3,
            PSA,
            SCDown,
            C2fCIB,
            A2C2f,
            C3FFTFixed,
            C3A2
        }
    )
    repeat_modules = frozenset(  # modules with 'repeat' arguments
        {
            BottleneckCSP,
            C1,
            C2,
            C2f,
            C3k2,
            C2fAttn,
            C3,
            C3TR,
            C3Ghost,
            C3x,
            RepC3,
            C2fPSA,
            C2fCIB,
            C2PSA,
            A2C2f,
            C3FFTFixed,
            C3A2
        }
    )
    for i, (f, n, m, args) in enumerate(d["backbone"] + d["head"]):  # from, number, module, args
        m = (
            getattr(torch.nn, m[3:])
            if "nn." in m
            else getattr(__import__("torchvision").ops, m[16:])
            if "torchvision.ops." in m
            else globals()[m]
        )  # get module
        for j, a in enumerate(args):
            if isinstance(a, str):
                with contextlib.suppress(ValueError):
                    args[j] = locals()[a] if a in locals() else ast.literal_eval(a)
        n = n_ = max(round(n * depth), 1) if n > 1 else n  # depth gain
        if m in base_modules:
            c1, c2 = ch[f], args[0]
            if c2 != nc:  # if c2 != nc (e.g., Classify() output)
                c2 = make_divisible(min(c2, max_channels) * width, 8)
            if m is C2fAttn:  # set 1) embed channels and 2) num heads
                args[1] = make_divisible(min(args[1], max_channels // 2) * width, 8)
                args[2] = int(max(round(min(args[2], max_channels // 2 // 32)) * width, 1) if args[2] > 1 else args[2])

            args = [c1, c2, *args[1:]]
            if m in repeat_modules:
                args.insert(2, n)  # number of repeats
                n = 1
            if m is C3k2:  # for M/L/X sizes
                legacy = False
                if scale in "mlx":
                    args[3] = True
            if m is A2C2f:
                legacy = False
                if scale in "lx":  # for L/X sizes
                    args.extend((True, 1.2))
            if m is C2fCIB:
                legacy = False
        elif m is AIFI:
            args = [ch[f], *args]
        elif m in frozenset({HGStem, HGBlock}):
            c1, cm, c2 = ch[f], args[0], args[1]
            args = [c1, cm, c2, *args[2:]]
            if m is HGBlock:
                args.insert(4, n)  # number of repeats
                n = 1
        elif m is ResNetLayer:
            c2 = args[1] if args[3] else args[1] * 4
        elif m is torch.nn.BatchNorm2d:
            args = [ch[f]]
        elif m is Concat:
            c2 = sum(ch[x] for x in f)
        elif m is CGAFusionFixed:
            # CGAFusion需要两个输入特征图
            if isinstance(f, list) and len(f) == 2:
                # 计算输出通道数
                c1 = ch[f[0]]  # 第一个输入特征的通道数
                c2 = args[0] if args else c1  # 使用args中的第一个参数作为输出通道数
                args = [c1, c2]  # CGAFusion需要c1, c2参数
            else:
                raise ValueError(f"CGAFusion requires exactly 2 input features, got {f}")
        elif m in frozenset(
            {
                Detect,
                WorldDetect,
                YOLOEDetect,
                Segment,
                Segment26,
                YOLOESegment,
                YOLOESegment26,
                Pose,
                Pose26,
                OBB,
                OBB26,
            }
        ):
            args.extend([reg_max, end2end, [ch[x] for x in f]])
            if m is Segment or m is YOLOESegment or m is Segment26 or m is YOLOESegment26:
                args[2] = make_divisible(min(args[2], max_channels) * width, 8)
            if m in {Detect, YOLOEDetect, Segment, Segment26, YOLOESegment, YOLOESegment26, Pose, Pose26, OBB, OBB26}:
                m.legacy = legacy
        elif m is v10Detect:
            args.append([ch[x] for x in f])
        elif m is ImagePoolingAttn:
            args.insert(1, [ch[x] for x in f])  # channels as second arg
        elif m is RTDETRDecoder:  # special case, channels arg must be passed in index 1
            args.insert(1, [ch[x] for x in f])
        elif m is CBLinear:
            c2 = args[0]
            c1 = ch[f]
            args = [c1, c2, *args[1:]]
        elif m is CBFuse:
            c2 = ch[f[-1]]
        elif m in frozenset({TorchVision, Index}):
            c2 = args[0]
            c1 = ch[f]
            args = [*args[1:]]
        else:
            c2 = ch[f]

        m_ = torch.nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)  # module
        t = str(m)[8:-2].replace("__main__.", "")  # module type
        m_.np = sum(x.numel() for x in m_.parameters())  # number params
        m_.i, m_.f, m_.type = i, f, t  # attach index, 'from' index, type
        if verbose:
            LOGGER.info(f"{i:>3}{f!s:>20}{n_:>3}{m_.np:10.0f}  {t:<45}{args!s:<30}")  # print
        save.extend(x % i for x in ([f] if isinstance(f, int) else f) if x != -1)  # append to savelist
        layers.append(m_)
        if i == 0:
            ch = []
        ch.append(c2)
    return torch.nn.Sequential(*layers), sorted(save)


def yaml_model_load(path):
    """Load a YOLOv8 model from a YAML file.

    Args:
        path (str | Path): Path to the YAML file.

    Returns:
        (dict): Model dictionary.
    """
    path = Path(path)
    if path.stem in (f"yolov{d}{x}6" for x in "nsmlx" for d in (5, 8)):
        new_stem = re.sub(r"(\d+)([nslmx])6(.+)?$", r"\1\2-p6\3", path.stem)
        LOGGER.warning(f"Ultralytics YOLO P6 models now use -p6 suffix. Renaming {path.stem} to {new_stem}.")
        path = path.with_name(new_stem + path.suffix)

    unified_path = re.sub(r"(\d+)([nslmx])(.+)?$", r"\1\3", str(path))  # i.e. yolov8x.yaml -> yolov8.yaml
    yaml_file = check_yaml(unified_path, hard=False) or check_yaml(path)
    d = YAML.load(yaml_file)  # model dict
    d["scale"] = guess_model_scale(path)
    d["yaml_file"] = str(path)
    return d


def guess_model_scale(model_path):
    """Extract the size character n, s, m, l, or x of the model's scale from the model path.

    Args:
        model_path (str | Path): The path to the YOLO model's YAML file.

    Returns:
        (str): The size character of the model's scale (n, s, m, l, or x).
    """
    try:
        return re.search(r"yolo(e-)?[v]?\d+([nslmx])", Path(model_path).stem).group(2)
    except AttributeError:
        return ""


def guess_model_task(model):
    """Guess the task of a PyTorch model from its architecture or configuration.

    Args:
        model (torch.nn.Module | dict): PyTorch model or model configuration in YAML format.

    Returns:
        (str): Task of the model ('detect', 'segment', 'classify', 'pose', 'obb').
    """

    def cfg2task(cfg):
        """Guess from YAML dictionary."""
        m = cfg["head"][-1][-2].lower()  # output module name
        if m in {"classify", "classifier", "cls", "fc"}:
            return "classify"
        if "detect" in m:
            return "detect"
        if "segment" in m:
            return "segment"
        if "pose" in m:
            return "pose"
        if "obb" in m:
            return "obb"

    # Guess from model cfg
    if isinstance(model, dict):
        with contextlib.suppress(Exception):
            return cfg2task(model)
    # Guess from PyTorch model
    if isinstance(model, torch.nn.Module):  # PyTorch model
        for x in "model.args", "model.model.args", "model.model.model.args":
            with contextlib.suppress(Exception):
                return eval(x)["task"]  # nosec B307: safe eval of known attribute paths
        for x in "model.yaml", "model.model.yaml", "model.model.model.yaml":
            with contextlib.suppress(Exception):
                return cfg2task(eval(x))  # nosec B307: safe eval of known attribute paths
        for m in model.modules():
            if isinstance(m, (Segment, YOLOESegment)):
                return "segment"
            elif isinstance(m, Classify):
                return "classify"
            elif isinstance(m, Pose):
                return "pose"
            elif isinstance(m, OBB):
                return "obb"
            elif isinstance(m, (Detect, WorldDetect, YOLOEDetect, v10Detect)):
                return "detect"

    # Guess from model filename
    if isinstance(model, (str, Path)):
        model = Path(model)
        if "-seg" in model.stem or "segment" in model.parts:
            return "segment"
        elif "-cls" in model.stem or "classify" in model.parts:
            return "classify"
        elif "-pose" in model.stem or "pose" in model.parts:
            return "pose"
        elif "-obb" in model.stem or "obb" in model.parts:
            return "obb"
        elif "detect" in model.parts:
            return "detect"

    # Unable to determine task from model
    LOGGER.warning(
        "Unable to automatically guess model task, assuming 'task=detect'. "
        "Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'."
    )
    return "detect"  # assume detect

In [10]:
!yolo task=detect mode=val \
model=/content/ultralytics/ultralytics/best_weights/final.pt \
data=/content/ultralytics/DeepPCB_YOLO/data.yaml \
imgsz=640 \
batch=16

Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv5_C3FFTFixed_C3A2_CGAFusionFixed summary: 196 layers, 2,452,176 parameters, 0 gradients, 7.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 633.3±376.5 MB/s, size: 22.2 KB)
val: Scanning /content/ultralytics/DeepPCB_YOLO/labels/val... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 1.1Kit/s 0.2s
val: New cache created: /content/ultralytics/DeepPCB_YOLO/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.7it/s 4.9s
                   all        200       1388      0.959      0.901       0.97      0.752
                  open        181        253      0.983      0.906      0.974      0.675
                 short        149        198       0.94      0.869       0.95      0.655
             mousebite        174        284      0.942      0.917       0.97      0.738
                  spur        16

In [11]:
# 安装必要依赖
!pip install fastapi uvicorn python-multipart pillow ultralytics pyngrok

In [12]:
# 单元格 1: 清理旧进程并安装依赖
import os
import sys
import subprocess
import time
import signal

print("🔧 清理旧进程...")

# 查找并终止占用8000端口的进程
try:
    # 使用fuser命令查找占用端口的进程
    result = subprocess.run(["lsof", "-ti:8000"], capture_output=True, text=True)
    if result.stdout:
        pids = result.stdout.strip().split()
        for pid in pids:
            try:
                os.kill(int(pid), signal.SIGTERM)
                print(f"  已终止进程 {pid}")
            except:
                pass
        time.sleep(2)  # 等待进程完全终止
except:
    pass

print("📦 安装依赖包...")
packages = [
    'ultralytics',
    'fastapi',
    'uvicorn[standard]',
    'python-multipart',
    'pillow',
    'opencv-python-headless',
    'nest-asyncio',
    'aiofiles'
]

for package in packages:
    try:
        __import__(package.replace('-', '_').split('[')[0])
        print(f"  ✅ {package} 已安装")
    except ImportError:
        print(f"  📥 正在安装 {package}...")
        subprocess.run([sys.executable, "-m", "pip", "install", package, "-q"], capture_output=True)
        print(f"  ✅ {package} 安装完成")

print("✅ 清理和安装完成")

🔧 清理旧进程...
📦 安装依赖包...
  ✅ ultralytics 已安装
  ✅ fastapi 已安装
  ✅ uvicorn[standard] 已安装
  ✅ python-multipart 已安装
  📥 正在安装 pillow...
  ✅ pillow 安装完成
  📥 正在安装 opencv-python-headless...
  ✅ opencv-python-headless 安装完成
  ✅ nest-asyncio 已安装
  ✅ aiofiles 已安装
✅ 清理和安装完成


In [13]:
# 单元格 2: 检查模型文件
import os
import glob

print("🔍 检查模型文件...")

# 搜索所有.pt文件
all_pt_files = glob.glob("/content/**/*.pt", recursive=True)

found_models = []
for path in all_pt_files:
    if os.path.exists(path):
        file_size = os.path.getsize(path) / (1024 * 1024)  # MB
        found_models.append((path, file_size))
        print(f"  ✅ 找到模型: {os.path.basename(path)} ({file_size:.1f} MB)")

if not found_models:
    print("  ❌ 未找到模型文件，将使用演示模式")
    model_path = "/content/dummy_model.pt"
else:
    # 选择最大的模型文件
    found_models.sort(key=lambda x: x[1], reverse=True)
    model_path = found_models[0][0]
    print(f"  🎯 选择模型: {os.path.basename(model_path)}")

model_path_str = str(model_path).replace('\\', '/')
print(f"✅ 模型检查完成")

🔍 检查模型文件...
  ✅ 找到模型: final.pt (5.0 MB)
  🎯 选择模型: final.pt
✅ 模型检查完成


In [14]:
# 单元格 3: 创建简化的后端服务（修改版）
print("\n⚙️ 创建后端服务...")

# 创建目录
os.makedirs("/content/ultralytics", exist_ok=True)

# 创建简化版后端代码
backend_code = '''
import io
import os
import cv2
import numpy as np
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse, JSONResponse, RedirectResponse, FileResponse
import uvicorn
import base64
import nest_asyncio
from ultralytics import YOLO
from contextlib import asynccontextmanager
import time
import json
import csv
from datetime import datetime
import zipfile
import uuid

# 应用nest_asyncio以在notebook中运行
nest_asyncio.apply()

# 配置
MODEL_PATH = "''' + model_path_str + '''"
CLASS_NAMES = ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']
RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
model = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    """应用生命周期管理"""
    global model
    print("🚀 正在加载模型...")
    try:
        if os.path.exists(MODEL_PATH):
            model = YOLO(MODEL_PATH)
            print(f"✅ 模型加载成功")
        else:
            raise FileNotFoundError(f"模型文件不存在")
    except Exception as e:
        print(f"❌ 模型加载失败: {e}")
        # 创建虚拟模型用于演示
        class DummyModel:
            def __init__(self):
                self.names = CLASS_NAMES
            def __call__(self, *args, **kwargs):
                class DummyResult:
                    def __init__(self):
                        self.boxes = None
                        self.plot = lambda: np.zeros((640, 640, 3), dtype=np.uint8)
                return [DummyResult()]
        model = DummyModel()
        print("⚠️  使用演示模式")
    yield
    print("🛑 应用关闭")

# 创建FastAPI应用
app = FastAPI(lifespan=lifespan)

# CORS配置
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def save_detection_results(image_name, detections, annotated_image, conf_threshold, original_size):
    """保存检测结果到文件"""
    try:
        # 生成唯一ID和时间戳
        detection_id = str(uuid.uuid4())[:8]
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # 创建本次检测的目录
        base_name = os.path.splitext(image_name)[0]
        result_dir = os.path.join(RESULTS_DIR, f"{timestamp}_{detection_id}_{base_name}")
        os.makedirs(result_dir, exist_ok=True)

        # 保存标注后的图片
        result_img_path = os.path.join(result_dir, f"annotated_{base_name}.png")
        Image.fromarray(annotated_image).save(result_img_path)

        # 保存检测结果到JSON文件
        result_json = {
            "detection_id": detection_id,
            "timestamp": datetime.now().isoformat(),
            "original_image": image_name,
            "image_size": original_size,
            "confidence_threshold": conf_threshold,
            "total_defects": len(detections),
            "defects_by_class": {},
            "detections": detections
        }

        # 统计各类缺陷数量
        for defect in detections:
            cls_name = defect["class"]
            if cls_name not in result_json["defects_by_class"]:
                result_json["defects_by_class"][cls_name] = 0
            result_json["defects_by_class"][cls_name] += 1

        # 保存JSON文件
        json_path = os.path.join(result_dir, "detection_results.json")
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(result_json, f, ensure_ascii=False, indent=2)

        # 保存CSV文件（用于Excel打开）
        csv_path = os.path.join(result_dir, "detection_results.csv")
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            # 写入头部信息
            writer.writerow(["PCB Defect Detection Report"])
            writer.writerow(["Detection ID", detection_id])
            writer.writerow(["Timestamp", datetime.now().strftime("%Y-%m-%d %H:%M:%S")])
            writer.writerow(["Original Image", image_name])
            writer.writerow(["Image Size", f"{original_size[0]}x{original_size[1]}"])
            writer.writerow(["Confidence Threshold", conf_threshold])
            writer.writerow(["Total Defects", len(detections)])
            writer.writerow([])  # 空行

            # 写入缺陷分类统计
            writer.writerow(["Defect Statistics by Class"])
            writer.writerow(["Class", "Count", "Percentage"])
            total = len(detections) if detections else 1
            for cls_name, count in result_json["defects_by_class"].items():
                percentage = (count / total) * 100
                writer.writerow([cls_name, count, f"{percentage:.1f}%"])
            writer.writerow([])  # 空行

            # 写入详细检测结果
            writer.writerow(["Detailed Detection Results"])
            writer.writerow(["ID", "Class", "Confidence", "X1", "Y1", "X2", "Y2", "Width", "Height"])
            for i, defect in enumerate(detections):
                bbox = defect["bbox"]
                width = bbox[2] - bbox[0]
                height = bbox[3] - bbox[1]
                writer.writerow([
                    i+1,
                    defect["class"],
                    f"{defect['confidence']:.4f}",
                    bbox[0], bbox[1], bbox[2], bbox[3],
                    width, height
                ])

        # 保存TXT报告
        txt_path = os.path.join(result_dir, "detection_report.txt")
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write("=" * 60 + "\\n")
            f.write("PCB Defect Detection Report\\n")
            f.write("=" * 60 + "\\n\\n")
            f.write(f"Detection ID: {detection_id}\\n")
            f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\\n")
            f.write(f"Original Image: {image_name}\\n")
            f.write(f"Image Size: {original_size[0]} x {original_size[1]}\\n")
            f.write(f"Confidence Threshold: {conf_threshold}\\n")
            f.write(f"Total Defects Detected: {len(detections)}\\n\\n")

            f.write("Defect Statistics:\\n")
            f.write("-" * 40 + "\\n")
            for cls_name, count in result_json["defects_by_class"].items():
                f.write(f"  {cls_name}: {count} defect(s)\\n")

            if detections:
                f.write("\\nDetailed Defect Information:\\n")
                f.write("-" * 60 + "\\n")
                for i, defect in enumerate(detections):
                    f.write(f"Defect #{i+1}:\\n")
                    f.write(f"  Class: {defect['class']}\\n")
                    f.write(f"  Confidence: {defect['confidence']:.4f}\\n")
                    f.write(f"  Bounding Box: {defect['bbox']}\\n")
                    f.write(f"  Position: X={defect['bbox'][0]}-{defect['bbox'][2]}, Y={defect['bbox'][1]}-{defect['bbox'][3]}\\n\\n")

            f.write("\\n" + "=" * 60 + "\\n")
            f.write("Detection completed.\\n")
            f.write("=" * 60 + "\\n")

        # 创建ZIP文件（包含所有结果）
        zip_path = os.path.join(RESULTS_DIR, f"{timestamp}_{detection_id}_{base_name}.zip")
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(result_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, result_dir)
                    zipf.write(file_path, arcname)

        return {
            "result_dir": result_dir,
            "zip_path": zip_path,
            "image_path": result_img_path,
            "json_path": json_path,
            "csv_path": csv_path,
            "txt_path": txt_path,
            "detection_id": detection_id
        }

    except Exception as e:
        print(f"保存结果时出错: {e}")
        return None

# 简单首页
@app.get("/")
async def home():
    return RedirectResponse(url="/frontend")

# 健康检查
@app.get("/health")
async def health_check():
    return {
        "status": "healthy",
        "service": "pcb-defect-detection",
        "model_loaded": model is not None,
        "results_dir": RESULTS_DIR,
        "timestamp": time.time()
    }

# 检测接口
@app.post("/detect")
async def detect(file: UploadFile = File(...), conf_threshold: float = Form(0.5)):
    try:
        # 读取图片
        contents = await file.read()
        img = Image.open(io.BytesIO(contents)).convert("RGB")
        img_np = np.array(img)
        original_size = img.size

        # 运行检测
        results = model(img_np, conf=conf_threshold, verbose=False)

        # 处理结果
        if len(results) > 0:
            annotated_img = results[0].plot()
            annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

            # 提取检测结果
            detections = []
            boxes = results[0].boxes
            if boxes is not None:
                for box in boxes:
                    cls_idx = int(box.cls[0])
                    cls_name = CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else f"class_{cls_idx}"
                    detections.append({
                        "class": cls_name,
                        "confidence": float(box.conf[0]),
                        "bbox": box.xyxy[0].cpu().numpy().astype(int).tolist()
                    })

            # 保存检测结果
            save_info = save_detection_results(
                file.filename,
                detections,
                annotated_img,
                conf_threshold,
                original_size
            )

            # 转为base64用于前端显示
            buffer = io.BytesIO()
            Image.fromarray(annotated_img).save(buffer, format="PNG")
            img_base64 = base64.b64encode(buffer.getvalue()).decode()

            response_data = {
                "status": "success",
                "detections": detections,
                "detection_count": len(detections),
                "annotated_image": img_base64,
                "message": f"检测到 {len(detections)} 个缺陷",
                "save_info": {
                    "detection_id": save_info["detection_id"] if save_info else None,
                    "files_saved": True if save_info else False,
                    "zip_file": os.path.basename(save_info["zip_path"]) if save_info else None
                }
            }

            if save_info:
                response_data["save_info"]["download_url"] = f"/download/{os.path.basename(save_info['zip_path'])}"

            return response_data
        else:
            # 如果没有检测结果，返回原图
            buffer = io.BytesIO()
            img.save(buffer, format="PNG")
            img_base64 = base64.b64encode(buffer.getvalue()).decode()

            # 即使没有检测到缺陷也保存结果
            save_info = save_detection_results(
                file.filename,
                [],
                img_np,
                conf_threshold,
                original_size
            )

            response_data = {
                "status": "success",
                "detections": [],
                "detection_count": 0,
                "annotated_image": img_base64,
                "message": "未检测到缺陷",
                "save_info": {
                    "detection_id": save_info["detection_id"] if save_info else None,
                    "files_saved": True if save_info else False,
                    "zip_file": os.path.basename(save_info["zip_path"]) if save_info else None
                }
            }

            if save_info:
                response_data["save_info"]["download_url"] = f"/download/{os.path.basename(save_info['zip_path'])}"

            return response_data

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"检测失败: {str(e)}")

# 下载检测结果
@app.get("/download/{filename}")
async def download_results(filename: str):
    zip_path = os.path.join(RESULTS_DIR, filename)
    if os.path.exists(zip_path):
        return FileResponse(
            zip_path,
            media_type='application/zip',
            filename=f"pcb_detection_{filename}"
        )
    else:
        raise HTTPException(status_code=404, detail="文件不存在")

# 查看所有检测记录
@app.get("/results")
async def list_results():
    results = []
    for filename in os.listdir(RESULTS_DIR):
        if filename.endswith('.zip'):
            file_path = os.path.join(RESULTS_DIR, filename)
            stats = os.stat(file_path)
            results.append({
                "filename": filename,
                "size_mb": stats.st_size / (1024 * 1024),
                "created": datetime.fromtimestamp(stats.st_ctime).strftime("%Y-%m-%d %H:%M:%S"),
                "download_url": f"/download/{filename}"
            })

    return {
        "total_results": len(results),
        "results": sorted(results, key=lambda x: x["created"], reverse=True)
    }

# 前端页面
@app.get("/frontend", response_class=HTMLResponse)
async def frontend_page():
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>PCB缺陷检测</title>
        <meta charset="utf-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <style>
            * {
                margin: 0;
                padding: 0;
                box-sizing: border-box;
            }

            body {
                font-family: Arial, sans-serif;
                background: #f0f2f5;
                color: #333;
                line-height: 1.6;
                padding: 20px;
            }

            .container {
                max-width: 1200px;
                margin: 0 auto;
                background: white;
                border-radius: 10px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.1);
                padding: 30px;
            }

            .header {
                text-align: center;
                margin-bottom: 30px;
                padding-bottom: 20px;
                border-bottom: 2px solid #eee;
            }

            .header h1 {
                color: #2c3e50;
                margin-bottom: 10px;
                font-size: 2.2em;
            }

            .header p {
                color: #7f8c8d;
                font-size: 1.1em;
            }

            .controls {
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(300px, 1fr));
                gap: 20px;
                margin-bottom: 30px;
            }

            .control-group {
                background: #f8f9fa;
                padding: 20px;
                border-radius: 8px;
                border: 1px solid #e9ecef;
            }

            .control-group h3 {
                color: #495057;
                margin-bottom: 15px;
                font-size: 1.2em;
            }

            input[type="file"] {
                width: 100%;
                padding: 12px;
                border: 2px dashed #007bff;
                border-radius: 6px;
                background: white;
                cursor: pointer;
                transition: all 0.3s;
            }

            input[type="file"]:hover {
                border-color: #0056b3;
                background: #f8f9ff;
            }

            input[type="range"] {
                width: 100%;
                margin: 15px 0;
                -webkit-appearance: none;
                height: 8px;
                background: #dee2e6;
                border-radius: 4px;
                outline: none;
            }

            input[type="range"]::-webkit-slider-thumb {
                -webkit-appearance: none;
                width: 22px;
                height: 22px;
                background: #007bff;
                border-radius: 50%;
                cursor: pointer;
                border: 3px solid white;
                box-shadow: 0 2px 5px rgba(0,0,0,0.2);
            }

            #confValue {
                display: block;
                text-align: center;
                font-weight: bold;
                color: #007bff;
                font-size: 1.2em;
                margin-top: 10px;
            }

            .detect-btn {
                width: 100%;
                padding: 16px;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                border: none;
                border-radius: 8px;
                font-size: 1.1em;
                font-weight: bold;
                cursor: pointer;
                transition: transform 0.3s, box-shadow 0.3s;
                margin-top: 10px;
            }

            .detect-btn:hover {
                transform: translateY(-2px);
                box-shadow: 0 5px 15px rgba(0,0,0,0.2);
            }

            .detect-btn:disabled {
                background: #6c757d;
                cursor: not-allowed;
                transform: none;
                box-shadow: none;
            }

            .results {
                display: grid;
                grid-template-columns: 1fr 1fr;
                gap: 30px;
                margin-top: 30px;
            }

            .image-container, .detections-container {
                background: white;
                padding: 20px;
                border-radius: 8px;
                border: 1px solid #e9ecef;
            }

            .image-container h3, .detections-container h3 {
                color: #495057;
                margin-bottom: 15px;
                font-size: 1.2em;
            }

            #resultImg {
                width: 100%;
                max-width: 600px;
                border-radius: 6px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.1);
                display: none;
            }

            .placeholder {
                text-align: center;
                padding: 40px;
                color: #adb5bd;
            }

            .placeholder i {
                font-size: 3em;
                margin-bottom: 15px;
                display: block;
            }

            .detection-item {
                background: #f8f9fa;
                padding: 15px;
                border-radius: 6px;
                margin-bottom: 10px;
                border-left: 4px solid #007bff;
            }

            .detection-item h4 {
                color: #2c3e50;
                margin-bottom: 8px;
                font-size: 1.1em;
            }

            .detection-item p {
                color: #6c757d;
                margin-bottom: 5px;
                font-size: 0.95em;
            }

            .no-defects {
                text-align: center;
                padding: 30px;
                background: #d4edda;
                border-radius: 6px;
                color: #155724;
                border: 1px solid #c3e6cb;
            }

            /* 新增的保存结果样式 */
            .save-results {
                margin-top: 30px;
                background: #e8f4f8;
                padding: 20px;
                border-radius: 8px;
                border-left: 4px solid #2196F3;
                display: none;
            }

            .save-results h3 {
                color: #0c5460;
                margin-bottom: 15px;
                font-size: 1.2em;
            }

            .file-list {
                display: grid;
                grid-template-columns: repeat(auto-fill, minmax(200px, 1fr));
                gap: 15px;
                margin-top: 15px;
            }

            .file-item {
                background: white;
                padding: 15px;
                border-radius: 6px;
                border: 1px solid #bee5eb;
            }

            .file-item .file-icon {
                font-size: 2em;
                margin-bottom: 10px;
                color: #2196F3;
            }

            .download-btn {
                display: inline-block;
                margin-top: 15px;
                padding: 12px 24px;
                background: linear-gradient(135deg, #28a745 0%, #20c997 100%);
                color: white;
                border: none;
                border-radius: 6px;
                text-decoration: none;
                font-weight: bold;
                text-align: center;
                transition: all 0.3s;
                cursor: pointer;
            }

            .download-btn:hover {
                transform: translateY(-2px);
                box-shadow: 0 4px 12px rgba(40, 167, 69, 0.3);
            }

            .download-btn .btn-icon {
                margin-right: 8px;
            }

            @media (max-width: 768px) {
                .results {
                    grid-template-columns: 1fr;
                }

                .controls {
                    grid-template-columns: 1fr;
                }

                .file-list {
                    grid-template-columns: 1fr;
                }
            }

            .status-bar {
                background: #e8f4f8;
                padding: 15px;
                border-radius: 6px;
                margin: 20px 0;
                text-align: center;
                color: #0c5460;
                border: 1px solid #bee5eb;
            }

            .statistics {
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(120px, 1fr));
                gap: 10px;
                margin-top: 15px;
            }

            .stat-item {
                background: white;
                padding: 10px;
                border-radius: 6px;
                text-align: center;
                border: 1px solid #dee2e6;
            }

            .stat-value {
                font-size: 1.5em;
                font-weight: bold;
                color: #007bff;
            }

            .stat-label {
                font-size: 0.9em;
                color: #6c757d;
                margin-top: 5px;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <div class="header">
                <h1>🔍 PCB缺陷检测系统</h1>
                <p>上传PCB图像，系统将自动检测缺陷并保存结果</p>
                <div class="status-bar" id="statusBar">
                    系统就绪，请上传图片开始检测
                </div>
            </div>

            <div class="controls">
                <div class="control-group">
                    <h3>1. 上传图像</h3>
                    <input type="file" id="imageInput" accept="image/*" onchange="previewImage()">
                </div>

                <div class="control-group">
                    <h3>2. 设置置信度</h3>
                    <input type="range" id="confSlider" min="0.1" max="1" step="0.05" value="0.5">
                    <span id="confValue">0.5</span>
                </div>

                <div class="control-group">
                    <h3>3. 开始检测</h3>
                    <button class="detect-btn" onclick="detect()" id="detectBtn">
                        🚀 开始检测
                    </button>
                </div>
            </div>

            <div class="save-results" id="saveResults">
                <h3>📁 检测结果已保存</h3>
                <div id="resultsStats" class="statistics"></div>
                <div class="file-list" id="fileList"></div>
                <a href="#" id="downloadBtn" class="download-btn" style="display: none;">
                    <span class="btn-icon">📥</span> 下载完整结果包
                </a>
            </div>

            <div class="results">
                <div class="image-container">
                    <h3>检测结果可视化</h3>
                    <img id="resultImg" alt="检测结果">
                    <div class="placeholder" id="imagePlaceholder">
                        <div style="font-size: 48px;">🖼️</div>
                        <p>检测结果将在这里显示</p>
                    </div>
                </div>

                <div class="detections-container">
                    <h3>检测结果详情</h3>
                    <div id="detections">
                        <div class="placeholder">
                            <div style="font-size: 48px;">📊</div>
                            <p>检测结果将在这里显示</p>
                        </div>
                    </div>
                </div>
            </div>
        </div>

        <script>
            // 更新置信度显示
            const confSlider = document.getElementById('confSlider');
            const confValue = document.getElementById('confValue');

            confSlider.addEventListener('input', function() {
                confValue.textContent = this.value;
            });

            // 图片预览
            function previewImage() {
                const fileInput = document.getElementById('imageInput');
                const resultImg = document.getElementById('resultImg');
                const placeholder = document.getElementById('imagePlaceholder');
                const saveResults = document.getElementById('saveResults');

                if (fileInput.files && fileInput.files[0]) {
                    const reader = new FileReader();

                    reader.onload = function(e) {
                        resultImg.src = e.target.result;
                        resultImg.style.display = 'block';
                        placeholder.style.display = 'none';
                    };

                    reader.readAsDataURL(fileInput.files[0]);

                    // 隐藏之前的结果
                    saveResults.style.display = 'none';
                }
            }

            // 显示保存结果
            function showSaveResults(data) {
                const saveResults = document.getElementById('saveResults');
                const fileList = document.getElementById('fileList');
                const downloadBtn = document.getElementById('downloadBtn');
                const resultsStats = document.getElementById('resultsStats');

                if (data.save_info && data.save_info.files_saved) {
                    // 更新统计信息
                    let statsHtml = '';
                    if (data.detection_count > 0) {
                        // 统计各类缺陷数量
                        const defectCounts = {};
                        data.detections.forEach(det => {
                            if (!defectCounts[det.class]) {
                                defectCounts[det.class] = 0;
                            }
                            defectCounts[det.class]++;
                        });

                        // 生成统计HTML
                        statsHtml += `
                            <div class="stat-item">
                                <div class="stat-value">${data.detection_count}</div>
                                <div class="stat-label">总缺陷数</div>
                            </div>
                        `;

                        for (const [cls, count] of Object.entries(defectCounts)) {
                            statsHtml += `
                                <div class="stat-item">
                                    <div class="stat-value">${count}</div>
                                    <div class="stat-label">${cls}</div>
                                </div>
                            `;
                        }
                    }

                    resultsStats.innerHTML = statsHtml;

                    // 显示文件列表
                    const files = [
                        { icon: '🖼️', name: '标注图像', type: 'annotated_image.png' },
                        { icon: '📊', name: '检测报告', type: 'detection_report.txt' },
                        { icon: '📈', name: 'CSV数据', type: 'detection_results.csv' },
                        { icon: '🗃️', name: 'JSON数据', type: 'detection_results.json' }
                    ];

                    let fileListHtml = '';
                    files.forEach(file => {
                        fileListHtml += `
                            <div class="file-item">
                                <div class="file-icon">${file.icon}</div>
                                <div><strong>${file.name}</strong></div>
                                <div style="font-size: 0.9em; color: #666;">${file.type}</div>
                            </div>
                        `;
                    });

                    fileList.innerHTML = fileListHtml;

                    // 设置下载按钮
                    if (data.save_info.download_url) {
                        downloadBtn.href = data.save_info.download_url;
                        downloadBtn.download = `pcb_detection_${data.save_info.zip_file}`;
                        downloadBtn.style.display = 'inline-block';
                    }

                    // 显示保存结果区域
                    saveResults.style.display = 'block';
                }
            }

            // 检测函数
            async function detect() {
                const fileInput = document.getElementById('imageInput');
                const detectBtn = document.getElementById('detectBtn');
                const statusBar = document.getElementById('statusBar');

                if (!fileInput.files || fileInput.files.length === 0) {
                    statusBar.innerHTML = '❌ 请先上传PCB图像！';
                    statusBar.style.background = '#f8d7da';
                    statusBar.style.color = '#721c24';
                    statusBar.style.borderColor = '#f5c6cb';
                    return;
                }

                const file = fileInput.files[0];
                const confThreshold = confSlider.value;

                // 更新状态
                detectBtn.disabled = true;
                detectBtn.innerHTML = '⏳ 检测中...';
                statusBar.innerHTML = '🔄 正在检测中，请稍候...';
                statusBar.style.background = '#fff3cd';
                statusBar.style.color = '#856404';
                statusBar.style.borderColor = '#ffeaa7';

                const formData = new FormData();
                formData.append('file', file);
                formData.append('conf_threshold', confThreshold);

                try {
                    const response = await fetch('/detect', {
                        method: 'POST',
                        body: formData
                    });

                    if (!response.ok) {
                        throw new Error(`HTTP错误: ${response.status}`);
                    }

                    const data = await response.json();

                    if (data.status === 'success') {
                        // 显示结果图片
                        const resultImg = document.getElementById('resultImg');
                        const placeholder = document.getElementById('imagePlaceholder');
                        resultImg.src = `data:image/png;base64,${data.annotated_image}`;
                        resultImg.style.display = 'block';
                        placeholder.style.display = 'none';

                        // 显示检测结果
                        const detectionsDiv = document.getElementById('detections');
                        if (data.detections && data.detections.length > 0) {
                            let html = '';
                            data.detections.forEach((det, index) => {
                                const confidencePercent = (det.confidence * 100).toFixed(1);
                                const confidenceColor = det.confidence > 0.8 ? '#28a745' :
                                                      det.confidence > 0.5 ? '#ffc107' : '#dc3545';
                                html += `
                                    <div class="detection-item">
                                        <h4>缺陷 #${index + 1}: ${det.class}</h4>
                                        <p>置信度: <span style="color: ${confidenceColor}; font-weight: bold;">${confidencePercent}%</span></p>
                                        <p>位置: [${det.bbox.join(', ')}]</p>
                                    </div>
                                `;
                            });
                            detectionsDiv.innerHTML = html;

                            // 更新状态
                            statusBar.innerHTML = `✅ 检测完成！发现 ${data.detection_count} 个缺陷`;
                            statusBar.style.background = '#d4edda';
                            statusBar.style.color = '#155724';
                            statusBar.style.borderColor = '#c3e6cb';
                        } else {
                            detectionsDiv.innerHTML = `
                                <div class="no-defects">
                                    <div style="font-size: 36px;">✅</div>
                                    <h4>未检测到任何缺陷</h4>
                                    <p>PCB板质量合格！</p>
                                </div>
                            `;

                            // 更新状态
                            statusBar.innerHTML = '✅ 检测完成！未发现缺陷';
                            statusBar.style.background = '#d4edda';
                            statusBar.style.color = '#155724';
                            statusBar.style.borderColor = '#c3e6cb';
                        }

                        // 显示保存结果
                        showSaveResults(data);

                    } else {
                        throw new Error(data.message || '检测失败');
                    }
                } catch (error) {
                    console.error('详细错误:', error);
                    statusBar.innerHTML = `❌ 检测失败: ${error.message}`;
                    statusBar.style.background = '#f8d7da';
                    statusBar.style.color = '#721c24';
                    statusBar.style.borderColor = '#f5c6cb';
                } finally {
                    // 恢复按钮状态
                    detectBtn.disabled = false;
                    detectBtn.innerHTML = '🚀 开始检测';
                }
            }

            // 添加键盘快捷键
            document.addEventListener('keydown', function(e) {
                if (e.key === 'Enter' && (e.ctrlKey || e.metaKey)) {
                    e.preventDefault();
                    detect();
                }
            });
        </script>
    </body>
    </html>
    """
    return HTMLResponse(content=html_content)

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info", access_log=False)

if __name__ == "__main__":
    run_server()
'''

# 删除旧的后端文件
old_files = ["/content/ultralytics/backend_final.py", "/content/ultralytics/backend.py"]
for file_path in old_files:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"  清理旧文件: {file_path}")

# 保存新的后端文件
with open("/content/ultralytics/backend.py", "w") as f:
    f.write(backend_code)

print("✅ 后端服务创建完成（已添加结果保存功能）")


⚙️ 创建后端服务...
✅ 后端服务创建完成（已添加结果保存功能）


In [15]:
# 单元格 4: 启动服务器并显示链接
import threading
import requests
import time
from IPython.display import display, HTML, clear_output
import sys

print("🚀 启动后端服务器...")

# 添加路径
sys.path.append('/content/ultralytics')

# 在后台线程中启动服务器
def run_server():
    try:
        import uvicorn
        from backend import app
        uvicorn.run(
            app,
            host="0.0.0.0",
            port=8000,
            log_level="warning",
            access_log=False
        )
    except Exception as e:
        print(f"❌ 服务器运行错误: {e}")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# 等待服务器启动
print("⏳ 等待服务器启动...")
max_wait = 40
server_started = False

for i in range(max_wait):
    try:
        response = requests.get("http://localhost:8000/health", timeout=2)
        if response.status_code == 200:
            print("✅ 后端服务器启动成功！")
            server_started = True
            break
    except Exception as e:
        if i == 0:
            print("  正在启动...")
    time.sleep(1)

    if i % 10 == 0 and i > 0:
        print(f"  等待中... ({i}/{max_wait})")

if not server_started:
    print("❌ 服务器启动失败")
    print("尝试检查端口占用情况...")
    try:
        import subprocess
        result = subprocess.run(["lsof", "-i:8000"], capture_output=True, text=True)
        if result.stdout:
            print("发现占用8000端口的进程:")
            print(result.stdout)
    except:
        pass
else:
    # 获取Colab代理地址
    print("\n🌐 获取访问地址...")

    colab_url = None
    try:
        from google.colab.output import eval_js
        colab_url = eval_js("google.colab.kernel.proxyPort(8000)")
        if colab_url:
            print("✅ 获取到Colab代理地址")
    except Exception as e:
        print(f"⚠️ 无法获取Colab代理: {e}")
        colab_url = None

    # 显示系统信息
    print("\n" + "=" * 60)
    print("🎉 PCB缺陷检测系统已启动！")
    print("=" * 60)

    model_name = os.path.basename(model_path_str)

    # 创建HTML显示信息
    html_content = f"""
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                padding: 30px; border-radius: 15px; color: white; margin: 20px 0;">
        <h2 style="text-align: center; margin-bottom: 30px;">🎯 PCB缺陷检测系统</h2>
        <div style="background: rgba(255,255,255,0.1); padding: 20px; border-radius: 10px;">
            <h3>🌐 访问地址:</h3>
    """

    if colab_url:
        html_content += f"""
            <div style="margin: 15px 0; padding: 15px; background: rgba(255,255,255,0.2); border-radius: 8px;">
                <strong>🔗 Colab代理地址 (推荐):</strong><br>
                <a href="{colab_url}" target="_blank" style="color: white; word-break: break-all; text-decoration: underline;">
                    {colab_url}
                </a>
            </div>
        """

    html_content += f"""
            <div style="margin: 15px 0; padding: 15px; background: rgba(255,255,255,0.2); border-radius: 8px;">
                <strong>🏠 本地地址:</strong><br>
                <a href="http://localhost:8000/frontend" target="_blank" style="color: white; word-break: break-all; text-decoration: underline;">
                    http://localhost:8000/frontend
                </a>
            </div>
        </div>
        <div style="margin-top: 30px; text-align: center;">
            <p><strong>💡 使用步骤:</strong></p>
            <ol style="text-align: left; display: inline-block; margin: 0 auto;">
                <li>点击上方链接打开系统</li>
                <li>上传PCB图片</li>
                <li>设置置信度（可选）</li>
                <li>点击"开始检测"按钮</li>
            </ol>
        </div>
        <div style="margin-top: 30px; font-size: 0.9em; opacity: 0.8; text-align: center;">
            <p>💾 模型: {model_name} | 🎯 支持6类缺陷检测</p>
        </div>
    </div>

    <div style="background: #e8f4f8; padding: 20px; border-radius: 10px; margin-top: 20px; border-left: 4px solid #2196F3;">
        <h4>📝 快速测试:</h4>
        <p><strong>1. 健康检查:</strong> <code>curl http://localhost:8000/health</code></p>
        <p><strong>2. 测试图片:</strong> 使用前端界面上传任意PCB或电子元件图片</p>
        <p><strong>3. 支持的缺陷类型:</strong> open, short, mousebite, spur, copper, pin-hole</p>
    </div>
    """

    display(HTML(html_content))

    print("\n📋 系统信息:")
    print(f"  模型: {model_name}")
    print(f"  端口: 8000")
    print(f"  状态: 运行中")

    print("\n⚠️  重要提示:")
    print("  • 如果无法访问，请确保Colab会话未过期")
    print("  • 检测完成后可下载结果图片")
    print("  • 按 Ctrl+C 停止服务器")

    print("\n🔄 系统运行中...")

    # 保持程序运行
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\n🛑 正在停止系统...")

🚀 启动后端服务器...
⏳ 等待服务器启动...
  正在启动...
🚀 正在加载模型...
✅ 模型加载成功
✅ 后端服务器启动成功！

🌐 获取访问地址...
✅ 获取到Colab代理地址

🎉 PCB缺陷检测系统已启动！



📋 系统信息:
  模型: final.pt
  端口: 8000
  状态: 运行中

⚠️  重要提示:
  • 如果无法访问，请确保Colab会话未过期
  • 检测完成后可下载结果图片
  • 按 Ctrl+C 停止服务器

🔄 系统运行中...

🛑 正在停止系统...


In [17]:
# 安装 ngrok
!pip install pyngrok -q

# 配置 ngrok Token（需要先去 ngrok 官网注册获取：https://ngrok.com/）
from pyngrok import ngrok
ngrok.set_auth_token("39CoZbKFfCpeah7U03VUWtgCgy6_4irXyBimv8MJjWb32ogyK")  # 替换为自己的 Token

# 暴露 8000 端口
public_url = ngrok.connect(8000)
print(f"✅ 公网访问地址: {public_url}")

✅ 公网访问地址: NgrokTunnel: "https://agnatical-homeostatic-modesta.ngrok-free.dev" -> "http://localhost:8000"
